In [45]:
# A bare Kaggle working directory has none of the package folders yet.
# Create them before the %%writefile cells below materialize the notebook modules.
from pathlib import Path
import numpy as np

for package_dir in (
    Path("src/common"),
    Path("src/extraction"),
    Path("src/normalization"),
    Path("src/retrieval"),
    Path("eval"),
):
    package_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
!pip install -q gptqmodel

In [46]:
%pip install --no-cache-dir --force-reinstall --no-deps "numpy==2.2.6"

In [47]:
import numpy as np

print(np.__version__)
print(np.__file__)
print(np.char.center(["abc"], 8))

In [48]:
%pip install --no-cache-dir --force-reinstall \
    "scipy==1.15.3" \
    "scikit-learn==1.6.1"

In [49]:
import numpy, scipy, sklearn, transformers

print(numpy.__version__)
print(scipy.__version__)
print(sklearn.__version__)
print(transformers.__version__)

In [50]:
%%writefile src/common/__init__.py
# extraction/normalization/... share this package for path constants.

Overwriting src/common/__init__.py


In [51]:
%%writefile src/common/paths.py
"""Shared path constants and idempotent-artifact helpers.

Not a config framework -- just the directory layout every stage needs to agree on,
plus a helper for the Kaggle "resume from artifact if session was interrupted" requirement
(see AGENTS.md environment constraints). Kept as plain constants/functions per AGENTS.md
Section 3 (no DI/config framework until there's a real need for one).
"""

import os
from pathlib import Path


def _detect_root() -> Path:
    if os.environ.get("VIFINQA_ROOT"):
        return Path(os.environ["VIFINQA_ROOT"])
    if Path("/kaggle/working").exists():
        return Path("/kaggle/working")
    return Path(__file__).resolve().parents[2]


ROOT = _detect_root()
DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
EVAL_DIR = ROOT / "eval"
DEV_QUESTIONS_DIR = EVAL_DIR / "dev_questions"

for _d in (RAW_DIR, INTERIM_DIR, PROCESSED_DIR, DEV_QUESTIONS_DIR):
    _d.mkdir(parents=True, exist_ok=True)


def artifact_exists(path) -> bool:
    """True if a stage artifact was already produced -- lets a cell be skipped on resume."""
    p = Path(path)
    return p.exists() and p.stat().st_size > 0


Overwriting src/common/paths.py


In [52]:
import sys
sys.path.insert(0, "src")

from common.paths import ROOT, DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, EVAL_DIR, DEV_QUESTIONS_DIR, artifact_exists

print("ROOT       :", ROOT)
print("RAW_DIR    :", RAW_DIR)
print("INTERIM_DIR:", INTERIM_DIR)
print("PROCESSED  :", PROCESSED_DIR)
print("Is Kaggle  :", str(ROOT) == "/kaggle/working")


ROOT       : /kaggle/working
RAW_DIR    : /kaggle/working/data/raw
INTERIM_DIR: /kaggle/working/data/interim
PROCESSED  : /kaggle/working/data/processed
Is Kaggle  : True


In [53]:
%pip install -q huggingface_hub

Note: you may need to restart the kernel to use updated packages.


## Checkpoint 1 Ã¢â‚¬â€ Load Data + EDA + Extraction

**Input:** the official dataset, `AIGuruTinix/ViFinQA` on Hugging Face. As of starting this
notebook, nobody in this repo had actually opened it Ã¢â‚¬â€ `docs/data.md`'s description was
unverified. Everything in this section is checked against the real thing, not assumed.

**Output:** (a) a small local cache of the corpus's question/metadata files plus a
representative sample of report `.txt` files, (b) an EDA summary (`data/interim/eda_sample_file_stats.csv`),
(c) a table-candidate extraction of that sample (`data/interim/extraction_sample_candidates.csv`),
(d) this section's written report for human review.

**Assumption boundary:** everything measured here is on a **hand-picked 8-file sample**
(diverse companies/sectors/years/naming conventions Ã¢â‚¬â€ see 1.1), *not* the full 1,973-report
corpus. Per `CONTEXT.md` Checkpoint instructions, full-corpus extraction is deliberately
deferred to Checkpoint 2, after this checkpoint is reviewed.

**Stop condition:** this section ends with a written summary and an `assert CHECKPOINT_1_APPROVED`
guard. Do not change that flag to `True` without the human review having actually happened.


### 1.1 Load & inspect the real dataset structure

`cpu-only`, `requires internet`. Downloads only the small non-report files first
(`README.md`, `code_stock.csv`, `questions/questions.jsonl`) plus the repo file *listing*
(not content) via the HF API, so we can print the real structure before writing any parsing
logic.


In [54]:
# requires internet (prep phase only)
import json
import shutil
import urllib.request
from pathlib import Path

from huggingface_hub import hf_hub_download

REPO_ID = "AIGuruTinix/ViFinQA"

meta_files = ["README.md", "code_stock.csv", "questions/questions.jsonl"]
local_meta_dir = RAW_DIR / "hf_meta"
local_meta_dir.mkdir(parents=True, exist_ok=True)

for f in meta_files:
    dest = local_meta_dir / Path(f).name
    if artifact_exists(dest):
        print(f"skip (cached): {dest}")
        continue
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset")
    shutil.copy(p, dest)
    print(f"downloaded {f} -> {dest} ({dest.stat().st_size} bytes)")


skip (cached): /kaggle/working/data/raw/hf_meta/README.md
skip (cached): /kaggle/working/data/raw/hf_meta/code_stock.csv
skip (cached): /kaggle/working/data/raw/hf_meta/questions.jsonl


In [55]:
# requires internet (prep phase only) -- file *listing* only, not content.
manifest_path = INTERIM_DIR / "hf_file_manifest.json"

if artifact_exists(manifest_path):
    with open(manifest_path, encoding="utf-8") as f:
        siblings = json.load(f)
    print(f"loaded cached manifest: {len(siblings)} files")
else:
    req = urllib.request.Request(f"https://huggingface.co/api/datasets/{REPO_ID}")
    with urllib.request.urlopen(req) as r:
        api_data = json.load(r)
    siblings = [s["rfilename"] for s in api_data["siblings"]]
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(siblings, f, ensure_ascii=False, indent=2)
    print(f"fetched and cached manifest: {len(siblings)} files")

report_files = [s for s in siblings if s.startswith("financial_statements/")]
companies = sorted({s.split("/")[1] for s in report_files})
non_standard = [s for s in report_files if "consolidated" not in s and "separate" not in s]

print(f"total files in dataset repo : {len(siblings)}")
print(f"report .txt files           : {len(report_files)}")
print(f"companies (tickers)         : {len(companies)}")
print(f"non-standard-named reports  : {len(non_standard)}  (no \'consolidated\'/\'separate\' in name -- \'aggregated\' or unlabeled)")
print("sample non-standard report paths:")
for s in non_standard[:5]:
    print(" ", s)


loaded cached manifest: 1977 files
total files in dataset repo : 1977
report .txt files           : 1973
companies (tickers)         : 100
non-standard-named reports  : 62  (no 'consolidated'/'separate' in name -- 'aggregated' or unlabeled)
sample non-standard report paths:
  financial_statements/ACV/2022/ACV_financial_statements_2022_aggregated/ACV_financial_statements_2022_aggregated_extracted.txt
  financial_statements/DTK/2025/DTK_financial_statements_2025_aggregated/DTK_financial_statements_2025_aggregated_extracted.txt
  financial_statements/EVF/2018/EVF_financial_statements_2018/EVF_financial_statements_2018_extracted.txt
  financial_statements/EVF/2019/EVF_financial_statements_2019/EVF_financial_statements_2019_extracted.txt
  financial_statements/EVF/2020/EVF_financial_statements_2020/EVF_financial_statements_2020_extracted.txt


**Real structure found** (matches the dataset's own `README.md`, which is the actual source
of truth Ã¢â‚¬â€ `docs/data.md` in this repo turned out to be a compatible but incomplete summary,
see note below):

- `financial_statements/{TICKER}/{YEAR}/{DOC_NAME}/{DOC_NAME}_extracted.txt` Ã¢â‚¬â€ **1,973** report
  files, **100** companies, years 2015-2025.
- `DOC_NAME` is usually `{TICKER}_financial_statements_{YEAR}_{consolidated|separate}`; 62
  files deviate (`..._aggregated`, or no suffix at all, e.g. `EVF_financial_statements_2018`).
- `questions/questions.jsonl` Ã¢â‚¬â€ **1,012** questions, `{id, question}` only, matching
  `CONTEXT.md`'s claim that no answers/train/dev are provided in this release.
- `code_stock.csv` Ã¢â‚¬â€ ticker Ã¢â€ â€™ Vietnamese company name, 101 rows (100 tickers + header Ã¢â‚¬â€ matches
  `code_stock.csv`'s own header row).

**Resolved (with user, 2026-08-20 Ã¢â‚¬â€ see `CHANGE_LOG.md`): `report_id` = parent directory name,
not "filename minus `.txt`".** `submission_guide.md`'s literal rule text ("final filename
component, `.txt` stripped") and its own worked example disagree once applied to the real
filenames: the actual files are named `<doc>_extracted.txt`, but the guide's example path
(`ocr_filter\AAAÃ‚Â5\AAA_financial_statements_2015_consolidated`) has neither a `.txt`
extension nor an `_extracted` suffix Ã¢â‚¬â€ it matches the *parent directory* name instead, which is
consistent with the `ocr_filter/` layout the dataset's own README describes as the organizers'
internal/reference corpus (distinct from the `financial_statements/` layout actually published
on Hugging Face). Verified across all 1,973 report files: `parent_directory_name +
"_extracted.txt" == filename` holds with **zero exceptions**, so report_id = parent directory
name is both well-defined and matches the guide's example exactly.
`src/extraction/parser.py`'s `derive_report_id()` implements this. This is our own interpretive
resolution, not an organizer-published clarification Ã¢â‚¬â€ revisit if the organizers publish one
that says otherwise.


### 1.2 Sample selection and content inspection

`cpu-only`, `requires internet`. Downloads a hand-picked, diverse 8-file sample Ã¢â‚¬â€ not a random
sample, a deliberately stratified one, chosen after inspecting the manifest above:

| file | why chosen |
|---|---|
| `AAA_..._2015_consolidated` | baseline: manufacturing/plastics, clean "MÃƒÂ£ sÃ¡Â»â€˜"-coded statement format |
| `ACB_..._2022_consolidated` | bank Ã¢â‚¬â€ different statement name ("BÃƒÂ¡o cÃƒÂ¡o tÃƒÂ¬nh hÃƒÂ¬nh tÃƒÂ i chÃƒÂ­nh", not "BÃ¡ÂºÂ£ng cÃƒÂ¢n Ã„â€˜Ã¡Â»â€˜i kÃ¡ÂºÂ¿ toÃƒÂ¡n"), different form code, no "MÃƒÂ£ sÃ¡Â»â€˜" column, multi-level rowspan/colspan headers |
| `VJC_..._2018_consolidated` | aviation; also the subject of a real question in the released set |
| `SCR_..._2017_separate` | real estate, "separate" (company-only) statement variant |
| `VSC_..._2017_separate` | logistics/ports, smaller report |
| `HT1_..._2019_consolidated` | cement, another real question's subject |
| `ACV_..._2022_aggregated` | one of the 62 non-standard-named reports |
| `EVF_..._2018` | non-standard: no consolidated/separate suffix at all |


In [56]:
# requires internet (prep phase only)
SAMPLE_FILES = [
    "financial_statements/AAA/2015/AAA_financial_statements_2015_consolidated/AAA_financial_statements_2015_consolidated_extracted.txt",
    "financial_statements/ACB/2022/ACB_financial_statements_2022_consolidated/ACB_financial_statements_2022_consolidated_extracted.txt",
    "financial_statements/VJC/2018/VJC_financial_statements_2018_consolidated/VJC_financial_statements_2018_consolidated_extracted.txt",
    "financial_statements/SCR/2017/SCR_financial_statements_2017_separate/SCR_financial_statements_2017_separate_extracted.txt",
    "financial_statements/VSC/2017/VSC_financial_statements_2017_separate/VSC_financial_statements_2017_separate_extracted.txt",
    "financial_statements/HT1/2019/HT1_financial_statements_2019_consolidated/HT1_financial_statements_2019_consolidated_extracted.txt",
    "financial_statements/ACV/2022/ACV_financial_statements_2022_aggregated/ACV_financial_statements_2022_aggregated_extracted.txt",
    "financial_statements/EVF/2018/EVF_financial_statements_2018/EVF_financial_statements_2018_extracted.txt",
]

sample_dir = RAW_DIR / "sample_reports"
sample_dir.mkdir(parents=True, exist_ok=True)
sample_local_paths = []
# report_id = parent directory name in the ORIGINAL corpus layout (see 1.1) -- computed from
# SAMPLE_FILES (the HF-relative paths), not from sample_local_paths. The local cache below is
# flattened (all files copied directly into sample_dir/), so deriving report_id from a local
# path's parent directory would silently give the wrong answer ("sample_reports" for every
# file). Keeping the two lists parallel and passing report_ids explicitly to extract_corpus
# avoids that trap -- see tests/extraction/test_parser.py::test_extract_corpus_report_ids_override_avoids_flattened_path_bug.
sample_report_ids = [Path(f).parent.name for f in SAMPLE_FILES]
for f in SAMPLE_FILES:
    dest = sample_dir / Path(f).name
    sample_local_paths.append(dest)
    if artifact_exists(dest):
        print(f"skip (cached): {dest.name}")
        continue
    p = hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset")
    shutil.copy(p, dest)
    print(f"downloaded {dest.name} ({dest.stat().st_size} bytes)")

skip (cached): AAA_financial_statements_2015_consolidated_extracted.txt
skip (cached): ACB_financial_statements_2022_consolidated_extracted.txt
skip (cached): VJC_financial_statements_2018_consolidated_extracted.txt
skip (cached): SCR_financial_statements_2017_separate_extracted.txt
skip (cached): VSC_financial_statements_2017_separate_extracted.txt
skip (cached): HT1_financial_statements_2019_consolidated_extracted.txt
skip (cached): ACV_financial_statements_2022_aggregated_extracted.txt
skip (cached): EVF_financial_statements_2018_extracted.txt


In [57]:
# no internet needed from here on -- everything below reads the local sample cached above.
for name in ["AAA_financial_statements_2015_consolidated_extracted.txt",
             "ACB_financial_statements_2022_consolidated_extracted.txt",
             "EVF_financial_statements_2018_extracted.txt"]:
    p = sample_dir / name
    text = p.read_text(encoding="utf-8", errors="replace")
    print("=" * 100)
    print(name, f"({len(text)} chars)")
    print("-" * 100)
    print(text[:600])
    print()


AAA_financial_statements_2015_consolidated_extracted.txt (118654 chars)
----------------------------------------------------------------------------------------------------
===== PAGE 1 =====
Signature Not Verified
Được ký bởi ĐOÀN VIỆT KHƯƠNG
Ngày ký: 02.03.2016 10:57

CÔNG TY CỔ PHẦN NHỰA VÀ MÔI TRƯỜNG XANH AN PHÁT
BÁO CÁO TÀI CHÍNH HỢP NHẤT ĐÃ ĐƯỢC KIỂM TOÁN
CHO NĂM TÀI CHÍNH KẾT THỨC NGÀY 31 THÁNG 12 NĂM 2015

Tháng 2 năm 2016

===== PAGE 2 =====
CÔNG TY CỔ PHẦN NHỰA VÀ MÔI TRƯỜNG XANH AN PHÁT

Lô CN11+CN12, cụm công nghiệp An Đồng, thị trấn Nam Sách, huyện Nam Sách, tỉnh Hải Dương

MỤC LỤC

<table><tr><td></td><td>TRANG</td></tr><tr><td>BÁO CÁO CỦA BAN TỔNG GIÁM ĐỐC</td><td>2 - 3</td></tr><tr><td>BÁO CÁO KIỂM TOÁN ĐỘC LẬP</td><td>4 - 5</td></tr><tr><td>BẢNG 

ACB_financial_statements_2022_consolidated_extracted.txt (278131 chars)
----------------------------------------------------------------------------------------------------
===== PAGE 1 =====
KPMG

M.S.C.N. 01
THAM

Ngân hàng

Confirms the report format directly: `===== PAGE N =====` page markers, narrative Vietnamese
text, and financial tables embedded inline as single-line `<table><tr><td>...</td></tr></table>`
HTML (with `rowspan`/`colspan` for multi-level headers, seen in the bank statement). This is
the real structure `data.md`'s brief description ("mixes narrative and tabular data inline")
undersold Ã¢â‚¬â€ there's no ambiguity about *where* the tables are; the ambiguity is entirely in the
narrative text's OCR quality and in inconsistent statement-type naming across sectors (see 1.4).


### 1.3 EDA on the raw sample

`cpu-only`. Measured on the 8-file stratified sample above Ã¢â‚¬â€ **not** the full corpus. File-size
distribution across the full 1,973-file corpus is available cheaply from the dataset's own
`README.md` (**~363 MiB total report text**, ~188 KiB/file average) without downloading
everything; this notebook does not re-derive that number, it's read directly off the dataset
card. Everything else below (OCR noise rate, table-type mix) genuinely needs file content, so
it's measured on the 8-file sample only, per Checkpoint 1's "small representative set first"
instruction.


In [58]:
import re
import unicodedata
import pandas as pd

NOISE_RUN_RE = re.compile(r"(.)( ?\1){15,}")   # a single character repeated 16+ times -- the
                                                 # OCR page-border artifact seen in ACB's file
LEAKED_TAG_RE = re.compile(r"<(fcel|ecel|nl|ucel|xcel)[^>]*>")  # non-HTML tokens leaked into
                                                                 # OCR text, seen once in-sample

file_stats = []
for p in sample_local_paths:
    content = p.read_text(encoding="utf-8", errors="replace")
    lines = content.splitlines()
    file_stats.append({
        "file": p.name,
        "n_lines": len(lines),
        "n_chars": len(content),
        "n_pages": content.count("===== PAGE"),
        "n_table_tags": content.count("<table>"),
        "n_ocr_noise_runs": sum(1 for l in lines if NOISE_RUN_RE.search(l)),
        "n_leaked_structure_tags": len(LEAKED_TAG_RE.findall(content)),
    })

eda_df = pd.DataFrame(file_stats)
eda_out = INTERIM_DIR / "eda_sample_file_stats.csv"
eda_df.to_csv(eda_out, index=False)
print(f"saved {eda_out}")
eda_df


saved /kaggle/working/data/interim/eda_sample_file_stats.csv


,file,n_lines,n_chars,n_pages,n_table_tags,n_ocr_noise_runs,n_leaked_structure_tags
0,AAA_financial_statements_2015_consolidated_ext...,1686,118654,43,47,0,0
1,ACB_financial_statements_2022_consolidated_ext...,2992,278131,99,116,65,1
2,VJC_financial_statements_2018_consolidated_ext...,1505,120187,54,67,11,0
3,SCR_financial_statements_2017_separate_extract...,1367,123750,58,73,0,0
4,VSC_financial_statements_2017_separate_extract...,970,74733,36,41,1,0
5,HT1_financial_statements_2019_consolidated_ext...,1464,122136,50,61,0,0
6,ACV_financial_statements_2022_aggregated_extra...,1546,133605,58,60,0,0
7,EVF_financial_statements_2018_extracted.txt,1780,151094,55,75,2,0


In [59]:
print("--- file-level distribution across the 8-file sample (NOT the full corpus) ---")
print(eda_df[["n_lines", "n_chars", "n_pages", "n_table_tags"]].describe().round(1))
print()
print("OCR noise-run lines (repeated-character border artifacts) total:", eda_df["n_ocr_noise_runs"].sum())
print("  -- concentrated in ACB (bank, 99-page report): all but a few occur there.")
print("Leaked non-HTML structure tags (e.g. <fcel>) total:", eda_df["n_leaked_structure_tags"].sum())


--- file-level distribution across the 8-file sample (NOT the full corpus) ---
       n_lines   n_chars  n_pages  n_table_tags
count      8.0       8.0      8.0           8.0
mean    1663.8  140286.2     56.6          67.5
std      588.8   59683.3     18.8          22.9
min      970.0   74733.0     36.0          41.0
25%     1439.8  119803.8     48.2          56.8
50%     1525.5  122943.0     54.5          64.0
75%     1709.5  137977.2     58.0          73.5
max     2992.0  278131.0     99.0         116.0

OCR noise-run lines (repeated-character border artifacts) total: 79
  -- concentrated in ACB (bank, 99-page report): all but a few occur there.
Leaked non-HTML structure tags (e.g. <fcel>) total: 1


**EDA findings that shaped the extraction design below:**

1. **OCR noise is real but so far localized to narrative/border text, not inside `<table>` spans.**
   Across all 8 files, 0 of the ~2,600 `<td>` cells inspected structurally broke on this. The
   noisiest file (ACB, a 99-page bank report) has 65 lines of repeated-glyph border artifacts
   (e.g. a full line of `"C C C C C C ..."`) and one leaked `<fcel>` token Ã¢â‚¬â€ both outside table
   markup. This is measured on 8 files; it is *not* a claim that no table ever contains such
   noise in the full 1,973-file corpus.
2. **Statement-type naming is not uniform.** Regular companies use "BÃ¡ÂºÂ¢NG CÃƒâ€šN Ã„ÂÃ¡Â»ÂI KÃ¡ÂºÂ¾ TOÃƒÂN"
   (balance sheet) with a numeric "MÃƒÂ£ sÃ¡Â»â€˜" line-item code column. Banks (ACB) use "BÃƒÂO CÃƒÂO
   TÃƒÅ’NH HÃƒÅ’NH TÃƒâ‚¬I CHÃƒÂNH" instead, with **no** "MÃƒÂ£ sÃ¡Â»â€˜" column Ã¢â‚¬â€ rows are labeled with Roman
   numerals/letters instead. Any downstream schema/normalization design that assumes one fixed
   statement-name vocabulary or one fixed code column will silently break for banks and
   insurers Ã¢â‚¬â€ this is the kind of "Vietnamese terminology variance" `CONTEXT.md` Ã‚Â§8 already
   warned is a schema-linking problem, not something extraction should try to collapse away.
3. **Most `<table>`-tagged content is not a primary statement.** Classifying each table by the
   nearest preceding section-header keyword (tracked as running state across lines, not just
   the immediate 1-3 lines before the table Ã¢â‚¬â€ see below) gives, over the 540 tables in the
   8-file sample: **70.7% notes/schedules, 12.2% balance sheet, 12.2% income statement, 1.5%
   cash flow, 3.3% unclassified.** This roughly matches the dataset README's mention of the
   companion corpus having "143,815 normalized tables" for 1,973 reports (Ã¢â€°Ë†73/report Ã¢â‚¬â€ close
   to this sample's 540/8 Ã¢â€°Ë† 67.5/report). Extraction still captures *all* of these as raw
   candidates per `AGENTS.md` Ã‚Â§2 Ã¢â‚¬â€ it's retrieval's job to narrow them down, not extraction's.


### 1.4 Extraction: design + prototype run on the sample

`cpu-only`. Design, in order of what the EDA above actually showed (not assumed):

- **Line position is exact, for free.** Every `<table>...</table>` span sits entirely on one
  physical line (verified: 0 exceptions across all 540 tables in the sample). So
  `relevant_tables`'s required "start line of the table in the original OCR .txt file"
  (`submission_guide.md` Ã‚Â§5) is just the 1-indexed line number of that line Ã¢â‚¬â€ no heuristic
  re-derivation, no separate table-boundary detector needed.
- **Parse the HTML leniently, not with regex.** `rowspan`/`colspan` (seen in bank statements)
  need real grid expansion to produce usable rectangular tables; regex can't do that safely.
  Python's stdlib `html.parser.HTMLParser` is used because OCR output can contain unexpected
  tags (leaked `<fcel>`-style tokens) that would make a strict parser reject the whole table Ã¢â‚¬â€
  we want to capture what's there and flag anomalies, not discard.
- **Extraction does not classify or filter.** It records a `section_header` field (diagnostic:
  nearest preceding statement-type heading) alongside each table candidate, but this is
  *context*, not a decision Ã¢â‚¬â€ per `AGENTS.md` Ã‚Â§2, judging relevance is retrieval's job.

The module lives at `src/extraction/parser.py` in the repo (imported normally when running
locally; the cell below materializes the same file so this notebook is self-contained on a
bare Kaggle kernel). It has unit tests at `tests/extraction/test_parser.py` against a
synthetic fixture (`tests/fixtures/SYN_..._extracted.txt`) per `AGENTS.md` Ã‚Â§6 Ã¢â‚¬â€ two real bugs
were caught by these tests during development, both fixed and now regression-tested:

1. An unclosed `<td>` before the next `<td>`: `html.parser.HTMLParser` does **not** implicitly
   close an unclosed cell tag the way a browser DOM parser would, so the previous cell's text
   was silently dropped until this was special-cased in `handle_starttag`.
2. `derive_report_id()` deriving from a *local* file path rather than the corpus-relative path:
   once `report_id` was redefined as the parent directory name (see the resolved question in
   1.1), a flattened local download cache (all sample files copied into one directory) would
   have silently produced the wrong report_id for every file. `extract_corpus()` now accepts an
   explicit `report_ids` list so callers pass report_id derived from the *original* corpus path,
   decoupling it from wherever the file happens to be cached locally Ã¢â‚¬â€ see 1.2's download cell.

Both are flagged here because they're exactly the kind of failure mode the full 1,973-file
corpus could still trigger even though the real 8-file sample didn't happen to exercise either.


In [60]:
%%writefile src/extraction/__init__.py
# see src/extraction/parser.py for the extraction-stage implementation.


Overwriting src/extraction/__init__.py


In [61]:
%%writefile src/extraction/parser.py
"""extraction stage: OCR .txt -> raw table candidates, with source line numbers preserved.

Scope boundary (AGENTS.md Section 2): this module identifies and structurally parses
<table>...</table> spans and records *where* they came from (file, physical line number,
page, nearby heading text). It does NOT decide which tables are "relevant" to any question,
and it does NOT drop tables that look like footnotes/schedules rather than primary financial
statements -- that judgment belongs to retrieval, not extraction.

Real, verified structural facts this module relies on (checked against 8 sample reports
spanning manufacturing, bank, "aggregated", and unlabeled document-name conventions --
see notebooks/vifinqa_pipeline.ipynb Checkpoint 1):
  - Each OCR report is a UTF-8 .txt file with page markers "===== PAGE N =====".
  - Every <table>...</table> span sits entirely on one physical line (verified: 0 exceptions
    across 540 tables in the 8-file sample). This makes "line position" trivial and exact --
    it's just the 1-indexed line number of that line, satisfying submission_guide.md's
    "start line of the table in the original OCR .txt file" requirement without any
    heuristic re-derivation.
  - Tables use only <table>/<tr>/<td> with optional rowspan/colspan attributes; no <th>.
    Multi-level headers (e.g. bank balance sheets) rely on rowspan/colspan and must be
    expanded into a dense grid to be usable downstream.
  - OCR noise (repeated-character border artifacts, occasional leaked non-HTML tags such as
    "<fcel>" from what looks like a table-structure-recognition model) occurs in the
    surrounding narrative text; in the 8-file sample it did not occur inside any <table> span.
    A lenient parser is still used deliberately, since this is a sample, not the full corpus.
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from html.parser import HTMLParser
from pathlib import Path
from typing import Optional

PAGE_RE = re.compile(r"^===== PAGE (\d+) =====\s*$")

# Heuristic section-header keywords, used only to attach diagnostic context to a table
# candidate (a "section_header" field) -- never to filter or drop candidates.
_SECTION_PATTERNS = [
    ("balance_sheet", re.compile(r"CÃƒâ€šN Ã„ÂÃ¡Â»ÂI KÃ¡ÂºÂ¾ TOÃƒÂN|TÃƒÅ’NH HÃƒÅ’NH TÃƒâ‚¬I CHÃƒÂNH", re.IGNORECASE)),
    ("income_statement", re.compile(r"KÃ¡ÂºÂ¾T QUÃ¡ÂºÂ¢ HOÃ¡ÂºÂ T Ã„ÂÃ¡Â»ËœNG|KÃ¡ÂºÂ¾T QUÃ¡ÂºÂ¢ KINH DOANH", re.IGNORECASE)),
    ("cash_flow", re.compile(r"LÃ†Â¯U CHUYÃ¡Â»â€šN TIÃ¡Â»â‚¬N", re.IGNORECASE)),
    ("notes", re.compile(r"THUYÃ¡ÂºÂ¾T MINH", re.IGNORECASE)),
]


def classify_section_header(text: Optional[str]) -> str:
    if not text:
        return "unclassified"
    for label, pat in _SECTION_PATTERNS:
        if pat.search(text):
            return label
    return "unclassified"


class TableGridParser(HTMLParser):
    """Lenient parser for a single <table>...</table> string.

    Deliberately lenient (HTMLParser, not a strict XML parser): OCR output can leak
    unexpected tags into surrounding text, and this stage's job is to capture what's there,
    not to reject anything that isn't perfectly well-formed HTML.
    """

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.rows: list[list[dict]] = []
        self._cur_row: Optional[list[dict]] = None
        self._cur_cell: Optional[list[str]] = None
        self._cur_cell_attrs: Optional[dict] = None
        self.warnings: list[str] = []

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "tr":
            self._cur_row = []
        elif tag in ("td", "th"):
            if self._cur_cell is not None:
                # unclosed <td>/<th> (OCR/table-markup artifact) -- html.parser does not
                # implicitly close it the way a browser DOM would, so do it ourselves or
                # the previous cell's text is silently dropped.
                self.warnings.append("unclosed <td>/<th> before next cell; auto-closed")
                self._flush_cell()
            self._cur_cell_attrs = attrs
            self._cur_cell = []

    def handle_endtag(self, tag):
        if tag == "tr" and self._cur_row is not None:
            self.rows.append(self._cur_row)
            self._cur_row = None
        elif tag in ("td", "th") and self._cur_cell is not None:
            self._flush_cell()

    def _flush_cell(self):
        text = "".join(self._cur_cell).strip()
        try:
            rowspan = int(self._cur_cell_attrs.get("rowspan", 1) or 1)
            colspan = int(self._cur_cell_attrs.get("colspan", 1) or 1)
        except ValueError:
            rowspan, colspan = 1, 1
            self.warnings.append(f"non-integer rowspan/colspan in cell {text!r}")
        if self._cur_row is None:
            self._cur_row = []
            self.warnings.append(f"cell {text!r} closed outside a <tr>; recovered")
        self._cur_row.append({"text": text, "rowspan": rowspan, "colspan": colspan})
        self._cur_cell = None
        self._cur_cell_attrs = None

    def handle_data(self, data):
        if self._cur_cell is not None:
            self._cur_cell.append(data)


def expand_grid(rows: list[list[dict]]) -> list[list[str]]:
    """Expand rowspan/colspan cells into a dense rectangular grid of strings."""
    grid: dict[tuple[int, int], str] = {}
    max_cols = 0
    for r_idx, row in enumerate(rows):
        c_idx = 0
        for cell in row:
            while (r_idx, c_idx) in grid:
                c_idx += 1
            text, rs, cs = cell["text"], cell["rowspan"], cell["colspan"]
            for dr in range(rs):
                for dc in range(cs):
                    grid[(r_idx + dr, c_idx + dc)] = text
            c_idx += cs
        max_cols = max(max_cols, c_idx)
    max_rows = (max((r for r, _ in grid), default=-1)) + 1
    return [[grid.get((r, c), "") for c in range(max_cols)] for r in range(max_rows)]


@dataclass
class TableCandidate:
    report_id: str
    source_path: str
    line_position: int  # 1-indexed physical line number of the <table> line in the .txt file
    page: Optional[int]
    section_header: str  # diagnostic only, see classify_section_header
    caption_context: list[str]  # up to 3 non-empty lines immediately preceding the table
    status: str  # "success" | "parsed_with_warnings" | "failed"
    warnings: list[str]
    n_rows: int
    n_cols: int
    grid: list[list[str]] = field(default_factory=list)


def derive_report_id(txt_path: str) -> str:
    """report_id per submission_guide.md, resolved (see CHANGE_LOG.md Checkpoint 1 entry).

    submission_guide.md's literal rule text ("final filename component, minus .txt") and its
    worked example disagree once applied to the real ViFinQA filenames: the actual files are
    named "<doc>_extracted.txt", but the guide's own example path
    (`ocr_filter\\AAA\\2015\\AAA_financial_statements_2015_consolidated`) has neither a `.txt`
    extension nor an `_extracted` suffix -- it matches the *parent directory* name instead. That
    example path shape matches the `ocr_filter/` corpus layout the dataset's own README
    describes as the organizers' internal/reference layout, distinct from the `financial_statements/`
    layout actually published on Hugging Face. Verified against all 1,973 report files in the
    published corpus: `parent_directory_name + "_extracted.txt" == filename` holds with zero
    exceptions, so report_id = parent directory name is both well-defined and consistent with
    the guide's example. `_extracted` is therefore this release's file-naming artifact, not
    part of the canonical report_id.
    """
    return Path(txt_path).parent.name


def extract_tables_from_file(txt_path: str, report_id: Optional[str] = None) -> list[TableCandidate]:
    path = Path(txt_path)
    rid = report_id or derive_report_id(str(path))
    with path.open(encoding="utf-8", errors="replace") as fh:
        lines = fh.readlines()

    results: list[TableCandidate] = []
    current_page: Optional[int] = None
    current_section: Optional[str] = None

    for i, line in enumerate(lines):
        stripped = line.strip()
        page_match = PAGE_RE.match(stripped)
        if page_match:
            current_page = int(page_match.group(1))
            continue

        if "<table>" not in stripped:
            for label, pat in _SECTION_PATTERNS:
                if pat.search(stripped):
                    current_section = stripped
                    break
            continue

        if "<table>" in stripped and "</table>" in stripped:
            line_no = i + 1
            caption_context: list[str] = []
            j = i - 1
            while j >= 0 and len(caption_context) < 3:
                s = lines[j].strip()
                if PAGE_RE.match(s):
                    break
                if s:
                    caption_context.insert(0, s)
                j -= 1

            status, warnings, grid = "success", [], []
            try:
                parser = TableGridParser()
                parser.feed(stripped)
                parser.close()
                warnings = parser.warnings
                grid = expand_grid(parser.rows)
                if warnings:
                    status = "parsed_with_warnings"
                if not grid or not grid[0]:
                    status = "failed"
                    warnings = warnings + ["empty grid after parse"]
            except Exception as e:  # noqa: BLE001 -- deliberately broad: record, don't crash the batch
                status = "failed"
                warnings = [f"{type(e).__name__}: {e}"]

            results.append(
                TableCandidate(
                    report_id=rid,
                    source_path=str(path),
                    line_position=line_no,
                    page=current_page,
                    section_header=classify_section_header(current_section),
                    caption_context=caption_context,
                    status=status,
                    warnings=warnings,
                    n_rows=len(grid),
                    n_cols=(max((len(r) for r in grid), default=0)),
                    grid=grid,
                )
            )

    return results


def extract_corpus(
    txt_paths: list[str], report_ids: Optional[list[str]] = None
) -> list[TableCandidate]:
    """Batch extraction over many files. Pure CPU/text work -- no model, no GPU.

    report_ids, if given, must be parallel to txt_paths and is passed through to
    extract_tables_from_file instead of relying on derive_report_id(local_path). This matters
    whenever the local cache/download layout doesn't mirror the original corpus's directory
    structure (derive_report_id relies on the *parent directory* name -- see its docstring --
    so a flattened local copy would silently produce a wrong report_id).
    """
    if report_ids is not None and len(report_ids) != len(txt_paths):
        raise ValueError("report_ids must be the same length as txt_paths if provided")
    all_results: list[TableCandidate] = []
    for i, p in enumerate(txt_paths):
        rid = report_ids[i] if report_ids is not None else None
        all_results.extend(extract_tables_from_file(p, report_id=rid))
    return all_results


Overwriting src/extraction/parser.py


In [62]:
import importlib
import extraction.parser as ep
importlib.reload(ep)

sample_paths_str = [str(p) for p in sample_local_paths]
candidates = ep.extract_corpus(sample_paths_str, report_ids=sample_report_ids)
print(f"total table candidates extracted from 8-file sample: {len(candidates)}")
print("distinct report_ids:", sorted(set(c.report_id for c in candidates)))


total table candidates extracted from 8-file sample: 540
distinct report_ids: ['AAA_financial_statements_2015_consolidated', 'ACB_financial_statements_2022_consolidated', 'ACV_financial_statements_2022_aggregated', 'EVF_financial_statements_2018', 'HT1_financial_statements_2019_consolidated', 'SCR_financial_statements_2017_separate', 'VJC_financial_statements_2018_consolidated', 'VSC_financial_statements_2017_separate']


In [63]:
candidates_df = pd.DataFrame([
    {
        "report_id": c.report_id,
        "line_position": c.line_position,
        "page": c.page,
        "section_header": c.section_header,
        "status": c.status,
        "n_warnings": len(c.warnings),
        "n_rows": c.n_rows,
        "n_cols": c.n_cols,
    }
    for c in candidates
])

cand_out = INTERIM_DIR / "extraction_sample_candidates.csv"
candidates_df.to_csv(cand_out, index=False)
print(f"saved {cand_out}")

status_counts = candidates_df["status"].value_counts()
print("\n--- extraction status, 540 tables, 8-file sample ---")
for status, n in status_counts.items():
    print(f"  {status:22s} {n:4d}  ({n/len(candidates_df):.1%})")

print("\n--- section_header distribution (diagnostic context only, not a filter) ---")
for label, n in candidates_df["section_header"].value_counts().items():
    print(f"  {label:20s} {n:4d}  ({n/len(candidates_df):.1%})")


saved /kaggle/working/data/interim/extraction_sample_candidates.csv

--- extraction status, 540 tables, 8-file sample ---
  success                 540  (100.0%)

--- section_header distribution (diagnostic context only, not a filter) ---
  notes                 382  (70.7%)
  income_statement       66  (12.2%)
  balance_sheet          66  (12.2%)
  unclassified           18  (3.3%)
  cash_flow               8  (1.5%)


In [64]:
warned_or_failed = [c for c in candidates if c.status != "success"]
if warned_or_failed:
    print(f"{len(warned_or_failed)} candidates with warnings/failures on the real sample:")
    for c in warned_or_failed[:10]:
        print(f"  {c.report_id} line={c.line_position} status={c.status} warnings={c.warnings}")
else:
    print("0 warnings/failures among the 540 real-sample table candidates.")
    print("(The unclosed-<td> failure mode described above was only caught by the synthetic")
    print(" fixture test, not by this real sample -- treat 100% as 'no counterexample yet',")
    print(" not 'proven robust', until the full-corpus run in Checkpoint 2.)")


0 warnings/failures among the 540 real-sample table candidates.
(The unclosed-<td> failure mode described above was only caught by the synthetic
 fixture test, not by this real sample -- treat 100% as 'no counterexample yet',
 not 'proven robust', until the full-corpus run in Checkpoint 2.)


### 1.5 Checkpoint 1 summary Ã¢â‚¬â€ **STOP for review**

**(a) Real data structure observed** (Section 1.1-1.2): `AIGuruTinix/ViFinQA` on HF is
1,973 OCR report `.txt` files (`financial_statements/{TICKER}/{YEAR}/{DOC}/{DOC}_extracted.txt`,
100 companies, 2015-2025) + 1,012 `{id, question}` pairs (no answers) + a tickerÃ¢â€ â€™name CSV. Report
text totals ~363 MiB per the dataset's own README (no download-everything needed to know this).
Each report mixes narrative OCR text with financial tables embedded as single-line
`<table><tr><td>...</td></tr></table>` HTML, `rowspan`/`colspan` used for multi-level headers.

**(b) Extraction method chosen and why:** line-scan for `<table>...</table>` spans (exact,
free line-position capture, verified against all 540 sample tables) + lenient
`html.parser.HTMLParser`-based grid parser with explicit rowspan/colspan expansion (handles
the bank multi-level-header case correctly, verified in unit tests) + a diagnostic
(non-filtering) `section_header` tag derived from running-state tracking of the nearest
statement-type heading. `report_id` is derived as the parent directory name (resolved decision,
see 1.1) and threaded through explicitly rather than re-derived from wherever a file is cached
locally. Extraction module: `src/extraction/parser.py`, tested at
`tests/extraction/test_parser.py` (8/8 passing, including regression tests for two real bugs
found during development).

**(c) Success rate on the 8-file sample:** 540/540 table candidates (100%) parsed structurally
without error or warning. This is a small, hand-picked sample, not a claim about the full
corpus Ã¢â‚¬â€ the unclosed-`<td>` bug (found via a synthetic fixture, not this real sample, and
now fixed + regression-tested) is a concrete reminder that "0 failures on 8 files" does not
mean "will not fail on 1,973 files."

**(d) Full-corpus (100 companies) time/resource estimate:**
- Download: ~363 MiB total (per dataset README) Ã¢â‚¬â€ a few minutes on typical Kaggle bandwidth,
  far below the ~20 GiB Kaggle working-directory budget. **No disk-quota concern.**
- Extraction compute: the 8-file sample (1.1 MB text, 540 tables) parsed in well under a
  second, pure CPU, no model. Linearly scaling to 363 MiB Ã¢â€ â€™ **on the order of a few minutes**
  for the full corpus, still CPU-only, no GPU needed for this stage.
- Expected table-candidate volume: 540/8 Ã¢â€°Ë† 67.5 tables/report Ãƒâ€” 1,973 reports Ã¢â€°Ë† **~133,000**
  candidates Ã¢â‚¬â€ consistent with (not derived from) the dataset README's mention of the
  companion corpus having 143,815 normalized tables for the same 1,973 reports.

**Open questions:** none blocking Checkpoint 2. The `report_id` discrepancy flagged during
Checkpoint 1 was resolved with the user (Ã‚Â§1.1) Ã¢â‚¬â€ revisit only if the organizers publish an
official clarification that contradicts it.

---
**Do not proceed to Checkpoint 2 (full-corpus extraction, normalization, retrieval) until this
checkpoint has been reviewed and approved.**


In [65]:
CHECKPOINT_1_APPROVED = True  # approved by the user in chat, 2026-08-20
assert CHECKPOINT_1_APPROVED, (
    "Checkpoint 1 is pending human review (see the markdown summary above). "
    "Do not proceed to Checkpoint 2 by editing this flag without that review happening."
)


## Checkpoint 2 Ã¢â‚¬â€ Normalization + Retrieval

**Input:** the full 1,973-report corpus (Checkpoint 1 worked from an 8-file sample only), the
extraction module approved in Checkpoint 1.

**Output:** (a) full-corpus extraction results, (b) a normalization module producing a unified
per-table schema (ticker/year/variant/searchable-text/unit-hints), (c) a working sparse (BM25)
retrieval baseline with real Precision/Recall/F2 on a hand-built dev set, (d) 2-3 embedding
model candidates for a dense option, presented for approval rather than hardcoded, per the
user's brief.

**Assumption boundary:** all P/R/F2 numbers in this section are measured on
`eval/dev_questions/dev_v1.jsonl` Ã¢â‚¬â€ **14 hand-verified questions out of the 1,012 in the
official release, not the official answer key.** See that file's own `README.md` for exactly
how each one was constructed and its documented limitations.

**Stop condition:** this section ends with a written summary and an `assert CHECKPOINT_2_APPROVED`
guard, plus an explicit choice for the user to make about which retrieval method (and, if dense
is chosen, which embedding model) to carry into Checkpoint 3.


### 2.1 Full-corpus extraction + normalization

`cpu-only`, `requires internet` for the download cell only. Downloads all 1,973 report files
(not just the 8-file sample), re-runs the same extraction module approved in Checkpoint 1 across
all of them, and adds a normalization module that turns each raw table candidate into a record
with structured company/year/variant metadata plus a searchable text blob for retrieval.


In [66]:
# requires internet (prep phase only) -- downloads the full report corpus, not just the
# 8-file Checkpoint 1 sample. huggingface_hub's own cache makes this idempotent: re-running
# this cell after an interrupted session only fetches what's missing.
from huggingface_hub import snapshot_download

snapshot_path = Path(
    snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        allow_patterns=["financial_statements/**"],
        max_workers=8,
    )
)
all_report_files = sorted((snapshot_path / "financial_statements").glob("*/*/*/*_extracted.txt"))
print(f"full corpus: {len(all_report_files)} report files at {snapshot_path}")


INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/AIGuruTinix/ViFinQA/revision/main "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/AIGuruTinix/ViFinQA/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/AIGuruTinix/ViFinQA/tree/main?expand=false&recursive=true&limit=1000&cursor=ZXlKbWFXeGxYMjVoYldVaU9pSm1hVzVoYm1OcFlXeGZjM1JoZEdWdFpXNTBjeTlIVmxJdk1qQXlNeUlzSW5SeVpXVmZiMmxrSWpvaU5EWXpZVEJrTnpRd1lqSXdNMlEwWkRFMVpUUXdNVEptWTJRM04yRXdObUZpTW1RME5XSmxNQ0o5OjEwMDA%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/AIGuruTinix/ViFinQA/tree/main?expand=false&recursive=true&limit=1000&cursor=ZXlKbWFXeGxYMjVoYldVaU9pSm1hVzVoYm1OcFlXeGZjM1JoZEdWdFpXNTBjeTlQUjBNdk1qQXhPQzlQUjBOZlptbHVZVzVqYVdGc1gzTjBZWFJsYldWdWRITmZNakF4T0Y5amIyNXpiMnhwWkdGMFpXUWlMQ0owY21WbFgyOXBaQ0k2SWpRMk0yRXdaRGMwTUdJeU1ETmtOR1F4TldVME1ERXlabU5rTnpkaE1EWmhZakprTkRWaVpUQWlmUT09OjIwMDA%3D "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/AIGuruTinix/ViFinQA/tr

Fetching ... files: 0it [00:00, ?it/s]

full corpus: 1973 report files at /root/.cache/huggingface/hub/datasets--AIGuruTinix--ViFinQA/snapshots/0450088ab22ec946f04f097586967ca405955b3b


In [67]:
%%writefile src/normalization/__init__.py
# see src/normalization/schema.py for the normalization-stage implementation.


Overwriting src/normalization/__init__.py


In [68]:
%%writefile src/normalization/schema.py
"""normalization stage: raw table candidates -> unified schema (company/year/line-item/unit).

Scope boundary (AGENTS.md Section 2): this module represents extraction's output in a form
retrieval and schema_linking can use -- it does not judge relevance and does not drop
candidates. "Unified schema" here means structured metadata (ticker/year/variant, parsed from
report_id -- see the real naming-pattern diversity documented in `parse_report_id`) plus a
per-table searchable text blob (joined cell text) for retrieval indexing, and a best-effort list
of detected currency-unit hints. It does NOT do full cell-level line-item parsing (mapping each
row to a canonical financial concept) -- that is schema_linking's job once a specific table has
already been retrieved for a specific question, not something to do speculatively for all
146K+ table candidates up front.
"""

from __future__ import annotations

import hashlib
import json
import re
import unicodedata
from dataclasses import dataclass
from typing import Optional

from extraction.parser import TableCandidate

# Real report_id naming patterns observed across the full 1,973-report corpus (see
# notebooks/vifinqa_pipeline.ipynb Checkpoint 2 / CHANGE_LOG.md):
#   TICKER_financial_statements_YEAR_{consolidated|separate|aggregated}      (1,887 reports)
#   TICKER_financial_statements_YEAR                                         (no variant stated)
#   TICKER_financial_statements_YEAR_{consolidated|separate}_{N}             (multi-part filing)
#   TICKER_YEAR_financial_statement_explanations[_N]                        (narrative-only doc)
#   TICKER_YEAR_explanatory_letters_{N}                                     (narrative-only doc)
# Token order and presence of "financial_statements" is NOT consistent, so year/variant are
# found by keyword/regex search rather than positional splitting.

_YEAR_RE = re.compile(r"(20\d{2})")
# real part suffixes observed are "_1"/"_2" (multi-part filings, e.g. HDB_..._separate_1) --
# restricted to 1-2 digits so a trailing 4-digit year (e.g. EVF_financial_statements_2018,
# which has no variant/part suffix at all) is never misread as a part number.
_PART_RE = re.compile(r"_(\d{1,2})$")

_VARIANT_KEYWORDS = [
    ("consolidated", "consolidated"),
    ("separate", "separate"),
    ("aggregated", "aggregated"),
    ("explanation", "explanatory"),
    ("explanatory", "explanatory"),
]

_UNIT_RE = re.compile(
    r"(nghìn\s*tỷ\s*đồng|nghìn\s*tỷ\s*VND|tỷ\s*đồng|tỷ\s*VND|triệu\s*đồng|triệu\s*VND|nghìn\s*đồng|nghìn\s*VND|VND)",
    re.IGNORECASE,
)

# Diagnostic-only foreign-currency hint, deliberately separate from _UNIT_RE/unit_scale_to_vnd:
# it does not participate in VND scaling (query_generation is untouched by this field) -- it
# exists so a table denominated in a non-VND currency is visible in the catalog instead of
# silently falling through unit detection built for VND-only phrasing.
_FOREIGN_CURRENCY_RE = re.compile(r"\bUSD\b|US\$|\bEUR\b|\bJPY\b|\bCNY\b|\bSGD\b", re.IGNORECASE)

# Retrieval-text builder version: bump this whenever build_schema_retrieval_text's field
# selection or ordering changes, so a persisted catalog can be checked against the code that
# produced it (see normalization/build_artifacts.py CATALOG_FIELDS).
RETRIEVAL_TEXT_VERSION = "schema_only_v1"

# 99th percentile of real corpus n_rows is 37 (measured on the full 146,243-table catalog);
# 200 is a generous cap that only trims the ~50 known OCR-pathology tables flagged in
# extraction_full_anomalies.csv (up to 1,008 rows), not ordinary tables.
MAX_ROW_LABELS = 200

# Matches extraction's own shape-anomaly threshold (diagnose_candidate_shape's max_cols=50):
# a wide/malformed table can have hundreds of distinct column header_paths, and unlike
# MAX_ROW_LABELS this had no cap until real BGE-M3 token-length EDA on the full corpus found a
# 29,740-token outlier traced to exactly this field (see CHANGE_LOG.md retrieval-representation
# entry). 99.9%+ of real tables have far fewer than 60 columns.
MAX_COLUMN_HEADERS = 60

# infer_period_labels scans every column of up to 6 header rows; an unusually wide table (the
# same OCR-pathology class that produces the n_cols anomalies in extraction_full_anomalies.csv)
# can therefore surface far more than the ~2-4 period labels an ordinary current/prior-period
# table has. Real corpus EDA still found a >8192-token outlier after MAX_ROW_LABELS/
# MAX_COLUMN_HEADERS alone -- traced to this field being the one remaining uncapped list.
MAX_PERIOD_LABELS = 40


def parse_report_id(report_id: str) -> tuple[str, Optional[int], str, Optional[int]]:
    """(ticker, year, variant, part) parsed from report_id. See module docstring for the
    real naming patterns this must handle -- it does not assume one fixed token layout."""
    ticker = report_id.split("_", 1)[0]

    year_match = _YEAR_RE.search(report_id)
    year = int(year_match.group(1)) if year_match else None

    variant = "unspecified"
    lower = report_id.lower()
    for keyword, label in _VARIANT_KEYWORDS:
        if keyword in lower:
            variant = label
            break

    part_match = _PART_RE.search(report_id)
    part = int(part_match.group(1)) if part_match else None

    return ticker, year, variant, part


def detect_units(text: str) -> list[str]:
    """Best-effort currency-unit hints found in a table's text. Diagnostic only -- does not
    convert or normalize values; schema_linking/query_generation must still confirm the unit
    for the specific line item they use."""
    seen = []
    for m in _UNIT_RE.finditer(text):
        u = m.group(1)
        if u not in seen:
            seen.append(u)
    return seen


def unit_scale_to_vnd(unit: str | None) -> int | None:
    """Return the VND multiplier for one explicit, normalized currency unit."""
    if not unit:
        return None
    text = "".join(
        char for char in unicodedata.normalize("NFD", unit.lower())
        if not unicodedata.combining(char)
    ).replace("đ", "d")
    if "nghin ty" in text:
        return 1_000_000_000_000
    if "ty" in text:
        return 1_000_000_000
    if "trieu" in text:
        return 1_000_000
    if "nghin" in text:
        return 1_000
    if "vnd" in text or "dong" in text:
        return 1
    return None


@dataclass
class NormalizedTable:
    report_id: str
    ticker: str
    year: Optional[int]
    variant: str
    part: Optional[int]
    line_position: int
    page: Optional[int]
    section_header: str
    searchable_text: str
    detected_units: list[str]
    n_rows: int
    n_cols: int
    status: str
    source_path: str
    caption_context: list[str]
    table_identity: str
    period_labels: list[dict]
    column_metadata: list[dict]
    row_labels: list[dict]
    grid: list[list[str]]
    retrieval_text: str
    retrieval_text_version: str
    currency_hint: Optional[str]
    header_depth: int
    content_hash: str


_PERIOD_RE = re.compile(
    r"(?:20\d{2}|31\s*[./-]\s*12|01\s*[./-]\s*01|số\s+(?:cuối|đầu)\s+(?:năm|kỳ)|năm\s+(?:nay|trước))",
    re.IGNORECASE,
)
_NUMERIC_CELL_RE = re.compile(r"^[\s()\-+.,%\d]+$")


def infer_period_labels(grid: list[list[str]], header_scan_rows: int = 6) -> list[dict]:
    """Preserve period-bearing header cells with their original grid coordinates."""
    labels = []
    for row_index, row in enumerate(grid[:header_scan_rows]):
        for column_index, value in enumerate(row):
            text = value.strip()
            if text and _PERIOD_RE.search(text):
                labels.append({"row_index": row_index, "column_index": column_index, "label": text})
    return labels


def infer_column_metadata(grid: list[list[str]], header_scan_rows: int = 6) -> list[dict]:
    """Record source-unit and period evidence per numeric column, not per table."""
    n_cols = max((len(row) for row in grid), default=0)
    columns = []
    for column_index in range(n_cols):
        header_cells = [
            row[column_index].strip() for row in grid[:header_scan_rows]
            if column_index < len(row) and row[column_index].strip()
            and not _NUMERIC_CELL_RE.fullmatch(row[column_index].strip())
        ]
        header_path = " | ".join(dict.fromkeys(header_cells))
        units = detect_units(header_path)
        unit = units[0] if units else None
        columns.append({
            "column_index": column_index,
            "header_path": header_path,
            "period_labels": [value for value in header_cells if _PERIOD_RE.search(value)],
            "source_unit": unit,
            "scale_to_vnd": unit_scale_to_vnd(unit),
        })
    return columns


def infer_row_labels(grid: list[list[str]]) -> list[dict]:
    """Keep one best-effort textual label per row, with coordinates and no canonical remapping."""
    labels = []
    for row_index, row in enumerate(grid):
        for column_index, value in enumerate(row):
            text = value.strip()
            if text and not _NUMERIC_CELL_RE.fullmatch(text):
                labels.append({"row_index": row_index, "column_index": column_index, "label": text})
                break
    return labels


def _dedupe_preserve_order(values) -> list[str]:
    return list(dict.fromkeys(v for v in values if v))


# OCR occasionally merges several source lines into one cell (observed directly in real
# corpus tables, e.g. VIF_financial_statements_2023_consolidated|363's row 1, a single label
# running to hundreds of characters). MAX_ROW_LABELS/MAX_COLUMN_HEADERS/MAX_PERIOD_LABELS cap
# *how many* labels are kept but not *how long* any single one is -- confirmed via real
# BGE-M3 token-length EDA: even after those count caps, one table still produced ~9,000 tokens,
# traced to exactly this. Matches schema_linking.linker._prompt_text's existing convention for
# the same underlying hazard (bounding one malformed cell so it cannot dominate a budget).
_MAX_LABEL_CHARS = 200


def _bounded(text: str, limit: int = _MAX_LABEL_CHARS) -> str:
    return text if len(text) <= limit else text[:limit - 1] + "…"


def detect_currency_hint(text: str) -> Optional[str]:
    """Best-effort non-VND currency mention. Diagnostic only -- see module note above _FOREIGN_CURRENCY_RE.
    Does not feed unit_scale_to_vnd or any query_generation logic."""
    match = _FOREIGN_CURRENCY_RE.search(text)
    return match.group(0).strip() if match else None


def infer_header_depth(grid: list[list[str]], header_scan_rows: int = 6) -> int:
    """Count of leading rows (within header_scan_rows) that look like pure header rows:
    every non-empty cell is non-numeric. Stops at the first row containing a numeric-looking
    cell (typically the first data row) or the first fully blank row. Diagnostic only -- it
    does not replace or gate infer_period_labels/infer_column_metadata's own fixed scan
    window; it exists to check whether header_scan_rows itself is well-chosen."""
    depth = 0
    for row in grid[:header_scan_rows]:
        cells = [cell.strip() for cell in row if cell.strip()]
        if cells and all(not _NUMERIC_CELL_RE.fullmatch(cell) for cell in cells):
            depth += 1
        else:
            break
    return depth


def content_hash(grid: list[list[str]]) -> str:
    """Diagnostic grid fingerprint for duplicate/near-duplicate incidence checks.

    Never used to drop or merge tables (AGENTS.md Section 2/4) -- purely inspectable metadata.
    """
    canonical = json.dumps(grid, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]


def build_schema_retrieval_text(
    ticker: str,
    company_name: str,
    year: Optional[int],
    variant: str,
    section_header: str,
    caption_context: list[str],
    detected_units: list[str],
    period_labels: list[dict],
    column_metadata: list[dict],
    row_labels: list[dict],
    *,
    max_row_labels: int = MAX_ROW_LABELS,
    max_column_headers: int = MAX_COLUMN_HEADERS,
    max_period_labels: int = MAX_PERIOD_LABELS,
) -> str:
    """Schema-oriented retrieval text: identity + structure + row labels, no bulk numeric
    cell dump. Row labels (e.g. "Doanh thu thuần", "Lợi nhuận sau thuế") are financial
    statements' strongest lexical/semantic signal and are deliberately included, deduplicated,
    not filtered out as "just cell text" -- see CHANGE_LOG.md retrieval-representation entry.

    Field order is front-loaded (identity -> structure -> row labels) so that truncation under
    a tokenizer's max sequence length drops the least-discriminative content (the row-label
    tail) first, not the identity/period fields that resolve company/year/variant ambiguity.
    """
    header_paths = [
        _bounded(v) for v in _dedupe_preserve_order(
            str(col.get("header_path", "")) for col in column_metadata if col.get("header_path")
        )[:max_column_headers]
    ]
    periods = [
        _bounded(v) for v in
        _dedupe_preserve_order(str(p["label"]) for p in period_labels)[:max_period_labels]
    ]
    rows = [
        _bounded(v) for v in
        _dedupe_preserve_order(str(r["label"]) for r in row_labels)[:max_row_labels]
    ]
    parts = [
        f"TICKER {ticker}.",
        f"COMPANY {company_name}." if company_name else "",
        f"YEAR {year}." if year else "",
        f"VARIANT {variant}." if variant and variant != "unspecified" else "",
        f"SECTION {section_header}." if section_header and section_header != "unclassified" else "",
        ("CAPTION " + " | ".join(_bounded(c) for c in caption_context) + ".") if caption_context else "",
        ("UNIT " + ", ".join(detected_units) + ".") if detected_units else "",
        ("PERIODS " + " | ".join(periods) + ".") if periods else "",
        ("COLUMNS " + " | ".join(header_paths) + ".") if header_paths else "",
        ("ROWS " + " | ".join(rows) + ".") if rows else "",
    ]
    return " ".join(p for p in parts if p)


def normalize_candidate(
    candidate: TableCandidate, company_by_ticker: Optional[dict[str, str]] = None
) -> NormalizedTable:
    ticker, year, variant, part = parse_report_id(candidate.report_id)
    company_name = (company_by_ticker or {}).get(ticker, "")
    cells = [cell.strip() for row in candidate.grid for cell in row if cell.strip()]
    searchable_text = " ".join(cells)
    table_identity = " | ".join(candidate.caption_context + [candidate.section_header])
    detected_units = detect_units(searchable_text + " " + " ".join(candidate.caption_context))
    period_labels = infer_period_labels(candidate.grid)
    column_metadata = infer_column_metadata(candidate.grid)
    row_labels = infer_row_labels(candidate.grid)
    return NormalizedTable(
        report_id=candidate.report_id,
        ticker=ticker,
        year=year,
        variant=variant,
        part=part,
        line_position=candidate.line_position,
        page=candidate.page,
        section_header=candidate.section_header,
        searchable_text=searchable_text,
        detected_units=detected_units,
        n_rows=candidate.n_rows,
        n_cols=candidate.n_cols,
        status=candidate.status,
        source_path=candidate.source_path,
        caption_context=list(candidate.caption_context),
        table_identity=table_identity,
        period_labels=period_labels,
        column_metadata=column_metadata,
        row_labels=row_labels,
        grid=[list(row) for row in candidate.grid],
        retrieval_text=build_schema_retrieval_text(
            ticker, company_name, year, variant, candidate.section_header,
            candidate.caption_context, detected_units, period_labels, column_metadata, row_labels,
        ),
        retrieval_text_version=RETRIEVAL_TEXT_VERSION,
        currency_hint=detect_currency_hint(searchable_text),
        header_depth=infer_header_depth(candidate.grid),
        content_hash=content_hash(candidate.grid),
    )


def normalize_corpus(
    candidates: list[TableCandidate], company_by_ticker: Optional[dict[str, str]] = None
) -> list[NormalizedTable]:
    return [normalize_candidate(c, company_by_ticker) for c in candidates]


def structured_record(table: NormalizedTable) -> dict:
    """Submission-oriented, JSON-serializable record retaining the complete source grid."""
    record = dict(table.__dict__)
    record["table_key"] = f"{table.report_id}|{table.line_position}"
    return record


def structured_record_json(table: NormalizedTable) -> str:
    return json.dumps(structured_record(table), ensure_ascii=False)


Overwriting src/normalization/schema.py


In [69]:
# no internet needed -- pure CPU; temp files + replace make this resumable.
import csv, os, time
import pandas as pd
import normalization.schema as ns
importlib.reload(ns)

# v6 changes the structured contract again (schema_only retrieval_text + currency_hint/
# header_depth/content_hash -- see CHANGE_LOG.md retrieval-representation entry); the old v5
# cell re-implemented row_labels/period_labels inline (with its own, buggy copy of the period
# regex, corrupted by a bad copy-paste) instead of calling normalization.schema directly. This
# now calls ns.normalize_corpus/ns.structured_record so this checkpoint can never drift from
# src/normalization/schema.py again, while keeping the same per-file file_stats_df contract
# the cells below depend on.
normalized_path = PROCESSED_DIR / "normalized_tables_entity_v6.csv"  # BM25/dense catalog
structured_path = PROCESSED_DIR / "normalized_tables_entity_v6.jsonl"  # complete grids/provenance
anomaly_path = INTERIM_DIR / "extraction_full_anomalies.csv"
full_candidates_path = INTERIM_DIR / "extraction_full_file_stats.csv"
required = [normalized_path, structured_path, anomaly_path, full_candidates_path]

CATALOG_FIELDS = [
    "table_key", "report_id", "ticker", "year", "variant", "part", "line_position",
    "page", "section_header", "table_identity", "searchable_text", "detected_units",
    "n_rows", "n_cols", "status", "source_path",
    "retrieval_text", "retrieval_text_version", "currency_hint", "header_depth", "content_hash",
]

if all(artifact_exists(p) for p in required):
    print("skip (cached): structured normalization artifacts already on disk")
else:
    company_by_ticker = {}
    with open(RAW_DIR / "hf_meta" / "code_stock.csv", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) >= 2:
                company_by_ticker[row[0]] = row[1]

    tmp_catalog = normalized_path.with_suffix(".csv.tmp")
    tmp_structured = structured_path.with_suffix(".jsonl.tmp")
    tmp_anomaly = anomaly_path.with_suffix(".csv.tmp")
    file_level, n_total, n_anomalies = [], 0, 0
    with open(tmp_catalog, "w", encoding="utf-8", newline="") as cf, open(tmp_structured, "w", encoding="utf-8") as sf, open(tmp_anomaly, "w", encoding="utf-8", newline="") as af:
        cw = csv.DictWriter(cf, fieldnames=CATALOG_FIELDS)
        aw = csv.DictWriter(af, fieldnames=["table_key", "report_id", "line_position", "source_path", "anomaly"])
        cw.writeheader(); aw.writeheader()
        for fpath in all_report_files:
            rid = ep.derive_report_id(str(fpath))
            source_path = fpath.relative_to(snapshot_path).as_posix()
            candidates = ep.extract_tables_from_file(str(fpath), report_id=rid)
            normalized = ns.normalize_corpus(candidates, company_by_ticker)
            file_level.append({
                "report_id": rid, "n_tables": len(candidates),
                "n_ok": sum(c.status == "success" for c in candidates),
                "n_warn": sum(c.status == "parsed_with_warnings" for c in candidates),
                "n_fail": sum(c.status == "failed" for c in candidates),
            })
            for candidate, nt in zip(candidates, normalized):
                record = ns.structured_record(nt)
                record["source_path"] = source_path  # snapshot-relative, not the local absolute path
                sf.write(json.dumps(record, ensure_ascii=False) + "\n")
                cw.writerow({
                    "table_key": record["table_key"], "report_id": nt.report_id, "ticker": nt.ticker,
                    "year": nt.year, "variant": nt.variant, "part": nt.part,
                    "line_position": nt.line_position, "page": nt.page, "section_header": nt.section_header,
                    "table_identity": nt.table_identity, "searchable_text": nt.searchable_text,
                    "detected_units": "|".join(nt.detected_units), "n_rows": nt.n_rows, "n_cols": nt.n_cols,
                    "status": nt.status, "source_path": source_path,
                    "retrieval_text": nt.retrieval_text, "retrieval_text_version": nt.retrieval_text_version,
                    "currency_hint": nt.currency_hint or "", "header_depth": nt.header_depth,
                    "content_hash": nt.content_hash,
                })
                for anomaly in ([f"n_rows>200:{candidate.n_rows}"] if candidate.n_rows > 200 else []) + ([f"n_cols>50:{candidate.n_cols}"] if candidate.n_cols > 50 else []):
                    aw.writerow({"table_key": record["table_key"], "report_id": rid, "line_position": candidate.line_position, "source_path": source_path, "anomaly": anomaly})
                    n_anomalies += 1
                n_total += 1
    for tmp, final in [(tmp_catalog, normalized_path), (tmp_structured, structured_path), (tmp_anomaly, anomaly_path)]:
        os.replace(tmp, final)
    pd.DataFrame(file_level).to_csv(full_candidates_path, index=False)
    print(f"saved {n_total} structured tables and {n_anomalies} non-filtering anomaly flags")
file_stats_df = pd.read_csv(full_candidates_path)


skip (cached): structured normalization artifacts already on disk


In [70]:
total_tables = int(file_stats_df["n_tables"].sum())
total_ok = int(file_stats_df["n_ok"].sum())
total_warn = int(file_stats_df["n_warn"].sum())
total_fail = int(file_stats_df["n_fail"].sum())
print(f"full corpus: {len(file_stats_df)} files, {total_tables} table candidates")
print(f"  success={total_ok} ({total_ok/total_tables:.2%})  warn={total_warn} ({total_warn/total_tables:.2%})  fail={total_fail} ({total_fail/total_tables:.2%})")
print()
zero_table_files = file_stats_df[file_stats_df["n_tables"] == 0]
print(f"files with 0 tables: {len(zero_table_files)}")
print(zero_table_files["report_id"].tolist())


full corpus: 1973 files, 146243 table candidates
  success=146243 (100.00%)  warn=0 (0.00%)  fail=0 (0.00%)

files with 0 tables: 8
['PRT_2020_financial_statement_explanations', 'PRT_2021_explanatory_letters_1', 'PRT_2021_explanatory_letters_2', 'PRT_2022_explanatory_letters_1', 'PRT_2022_explanatory_letters_2', 'PRT_2023_explanatory_letters_1', 'PRT_2023_explanatory_letters_2', 'PRT_2025_financial_statement_explanations_1']


**Full-corpus extraction findings** (all 1,973 reports Ã¢â‚¬â€ not the 8-file Checkpoint 1 sample):

- **146,243 table candidates, 100.00% structural success (0 warnings, 0 failures), ~191s CPU
  time.** Checkpoint 1 estimated "on the order of a few minutes" and "~133,000 candidates" from
  the 8-file sample Ã¢â‚¬â€ the real run came in close to that estimate (146,243, ~8% above the sample
  extrapolation) and stayed well within the time estimate. `status=success` means the markup
  parsed without an exception; it does not certify semantic table quality. A separate,
  non-filtering diagnostic records 57 full-corpus shape flags (53 `n_rows>200`, 4
  `n_cols>50`) for OCR pathologies/schema-linking review.
- 8 report files legitimately produced 0 tables Ã¢â‚¬â€ all 8 are `PRT` "explanatory letter"
  narrative-only documents (confirmed by direct inspection: no `<table>` markup present at all,
  not an extraction miss).
- Variant distribution parsed from `report_id` (see `parse_report_id`): 957 consolidated / 954
  separate / 7 aggregated / 55 other (38 unspecified + 17 explanatory) Ã¢â‚¬â€ this **matches the
  dataset's own README exactly** ("957 consolidated, 954 separate, 7 aggregated, and 55 other or
  unlabeled reports"), a strong independent cross-check that report_id parsing is correct across
  the whole corpus, not just the patterns spot-checked in Checkpoint 1.
- The "100% success on the sample doesn't prove full-corpus robustness" caveat from Checkpoint 1
  held up under the full run Ã¢â‚¬â€ but a real bug *was* still found during this checkpoint's own
  development (`derive_report_id` misreading a bare 4-digit year as a part number; see
  `CHANGE_LOG.md`), caught by a unit test before it reached this full-corpus run, not by the
  full-corpus run itself. Scale alone does not substitute for targeted tests.


### 2.2 Retrieval: method options

Three options, evaluated against what Checkpoint 1 actually found about this corpus (not a
default "BM25 is popular" choice):

**Option A Ã¢â‚¬â€ Sparse (BM25 over table cell text).** Implemented and evaluated below
(`src/retrieval/sparse.py`). Rationale: Checkpoint 1 found OCR noise concentrated in narrative
text, not inside `<table>` spans (0/540 sample tables affected) Ã¢â‚¬â€ so lexical term matching over
cell text starts from clean input. No model download, no eligibility question (BM25 has no
learned weights), no GPU.

**Two real weaknesses were found (not hypothesized) while first evaluating this, both fixed and
both worth stating plainly because they explain the numbers in 2.3:**
1. **Table cell text alone essentially never contains the company name or ticker.** A page
   header like "CÃƒâ€NG TY CÃ¡Â»â€ PHÃ¡ÂºÂ¦N NHÃ¡Â»Â°A VÃƒâ‚¬ MÃƒâ€I TRÃ†Â¯Ã¡Â»Å“NG XANH AN PHÃƒÂT" sits *outside* the `<table>` tag
   entirely (see Checkpoint 1's raw-file inspection). Plain BM25 over bare cell text scored
   **precision_macro = recall_macro = 0.000 on all 14 dev questions at every top_k tested** Ã¢â‚¬â€ it
   cannot tell *which company* a question is about at all. Fixed by enriching each indexed
   table's text with its `ticker` + Vietnamese company name (`code_stock.csv`) + `year`, all
   already available as normalization output.
2. **Enrichment alone was still not enough.** A natural-language Vietnamese question is mostly
   generic function words ("lÃƒÂ ", "cÃ¡Â»Â§a", "cÃƒÂ´ng ty", "nÃ„Æ’m"...) that appear in a large fraction of
   all 146,243 tables; summed across ~19 query tokens, they out-accumulated the 1-2 truly
   distinctive terms (ticker, company name) appearing only once in the correct table's short
   text Ã¢â‚¬â€ even a document containing *zero* matching identity terms could outscore the true
   match this way (confirmed by inspecting an actual case: `NVL_...|1166` outranked the correct
   `VJC_...|1179` despite not containing "vjc" or "vietjet" at all). Fixed with two standard,
   general IR techniques Ã¢â‚¬â€ a Vietnamese stopword filter (`DEFAULT_STOPWORDS`) and repeating the
   identity fields 5Ãƒâ€” per table (`build_enriched_document_text`) so their term frequency can
   compete with a large table's boilerplate volume. Neither is dev-set-specific curve-fitting;
   both are documented, parameterized, and testable (`tests/retrieval/test_sparse.py`).

**Option B Ã¢â‚¬â€ Dense embedding (open-weight, Ã¢â€°Â¤14B, pre-2026-06-01 candidate models below).**
Not implemented in this checkpoint Ã¢â‚¬â€ the user's brief is explicit that model choice needs
approval, not a hardcoded pick. Rationale for considering it at all: dense retrieval could help
with Vietnamese terminology variance (`CONTEXT.md` Ã‚Â§8) Ã¢â‚¬â€ e.g. matching "lÃƒÂ£i vay" against "chi
phÃƒÂ­ lÃƒÂ£i vay" or "lÃ¡Â»Â£i nhuÃ¡ÂºÂ­n sau thuÃ¡ÂºÂ¿" against "LNST" Ã¢â‚¬â€ that exact lexical BM25 cannot bridge.
Cost: needs a model download (prep-phase, internet-gated) and GPU-scale embedding of 146K+
table texts (Kaggle T4Ãƒâ€”2/P100 territory, not this local CPU session), plus the eligibility
bookkeeping BM25 doesn't need.

**Option C Ã¢â‚¬â€ Hybrid (sparse pre-filter + dense re-rank, or score fusion).** Not implemented.
Rationale: could combine BM25's free company/year disambiguation (once enriched, as above) with
dense retrieval's synonym tolerance for the line-item side of the query. Cost: two models/passes
instead of one, more moving parts to keep within F2's recall-favoring design (`CONTEXT.md` Ã‚Â§4)
without materializing precision as an afterthought.

**Recommendation for Checkpoint 3, pending your approval:** ship the enriched sparse baseline
now (it required no model choice and its dev-set numbers are below); treat dense/hybrid as a
follow-up experiment once a specific embedding model is approved, rather than blocking Checkpoint
3 on it.


**Dense retrieval candidate models** (open-weight, Ã¢â€°Â¤14B params, released before 2026-06-01,
verified via each model's real Hugging Face card Ã¢â‚¬â€ not from memory). Presented as options for
you to choose from; none is implemented yet.

| Candidate | Params | License | Released | Vietnamese support | Weights |
|---|---|---|---|---|---|
| `bkai-foundation-models/vietnamese-bi-encoder` | ~0.1B (PhoBERT-base backbone) | Apache 2.0 | Mar 2024 | **Explicitly Vietnamese-specific** Ã¢â‚¬â€ trained on Vietnamese MS MARCO/SQuAD translations + Zalo 2021 legal-retrieval data | `huggingface_hub.snapshot_download`, public, no gating |
| `intfloat/multilingual-e5-base` | XLM-RoBERTa-base architecture (12 layers, 768-dim) | MIT | 2024 | Claims "100 languages via XLM-R"; Vietnamese **not explicitly named** in the fetched model card Ã¢â‚¬â€ plausible but unconfirmed | same |
| `BAAI/bge-m3` | XLM-RoBERTa-large-scale architecture (1024-dim, up to 8192 tokens) | MIT | Feb 2024 | Claims "100+ working languages"; Vietnamese **not explicitly named** in the fetched model card Ã¢â‚¬â€ plausible but unconfirmed | same |

Trade-off as presented, not decided: the BKAI model is the only one with a *confirmed*
Vietnamese-training claim, and is by far the smallest (cheapest to embed 146K+ tables on
Kaggle's GPU quota) Ã¢â‚¬â€ but it was trained on legal/general text, not financial statements
specifically, and financial-domain OCR text is a domain shift either way. The two multilingual
options are larger, unconfirmed on Vietnamese specifically, but have stronger general retrieval
track records and (bge-m3) much longer context, which could matter for wide multi-column tables.
**This choice needs your input before it's implemented** (see 2.4).


### 2.3 Dev set + sparse retrieval evaluation

`cpu-only`. The dev set (`eval/dev_questions/dev_v1.jsonl`, 14 questions) was built by hand:
for each question, the target company/year was identified from the question text, the real
report file(s) were located in the full corpus downloaded above, and the specific table(s)
answering the question were found by keyword search over extraction's own cell text Ã¢â‚¬â€ then
visually confirmed before being recorded as ground truth. Full methodology, and the real
problems this process surfaced (a company-name-to-ticker ambiguity, "cÃƒÂ´ng ty mÃ¡ÂºÂ¹" ambiguity, and
the "adjacent years often share one table" structural finding used in 2.2 above) are documented
in `eval/dev_questions/README.md` Ã¢â‚¬â€ not repeated here.


In [71]:
%%writefile src/retrieval/__init__.py
# see src/retrieval/sparse.py for the sparse (BM25) retrieval implementation.


Overwriting src/retrieval/__init__.py


In [72]:
%%writefile src/retrieval/sparse.py
"""retrieval stage: question -> candidate table set, sparse (BM25) implementation.

Scope boundary (AGENTS.md Section 2): returns candidate tables with a score; does not
silently drop candidates below a hardcoded threshold -- `search()` returns every scored
candidate, and how many to keep (top_k) is an explicit, inspectable, tunable caller parameter,
not baked into this module. F2's recall-weighting (CONTEXT.md Section 4) means the natural lever
for tuning is top_k, and that choice belongs to whoever is calling this, informed by dev-set
measurement -- not a default buried here.

Not a "model" in the CONTEXT.md Section 3 eligibility sense: BM25 is a deterministic classical
IR ranking formula with no learned/pretrained weights, so the open-weight/size/release-date
constraints on pipeline models do not apply to it. That constraint becomes relevant only if/when
a dense embedding model is added (see the candidate list presented alongside this in Checkpoint
2 -- not implemented yet, pending approval).

Uses an inverted index (term -> posting list of doc indices) rather than scoring every document
for every query. At corpus scale (146K+ tables in the real corpus, see Checkpoint 2), a naive
"score every doc" search is impractically slow in pure Python; a query only needs to touch the
(much smaller) set of documents that share at least one term with it.
"""

from __future__ import annotations

import math
import re
from dataclasses import dataclass, field

_TOKEN_RE = re.compile(r"\w+", re.UNICODE)

# Measured necessity, not a guess: on the full 146K-table corpus (Checkpoint 2 dev-set
# evaluation), plain BM25 over enriched (ticker+company+year+cell-text) tables scored
# precision_macro=recall_macro=0.0 on all 14 dev questions at top_k=10. High-frequency
# Vietnamese function/boilerplate words ("lÃƒÂ ", "cÃ¡Â»Â§a", "cÃƒÂ´ng ty", "nÃ„Æ’m"...) appear in a large
# fraction of all 146K documents and, summed across a ~19-token natural-language question,
# out-accumulate the 1-2 truly distinctive terms (ticker, company name) that appear only once
# in a correct document's short text. Removing them raised recall_macro to 0.21 in isolation;
# combined with `identity_boost` below, to 0.46. This is standard IR practice (stopword
# removal), not a dev-set-specific hack -- but the list itself IS specific to this corpus's
# observed vocabulary (see CHANGE_LOG.md for the actual before/after numbers) and is a
# parameter precisely so it can be revised, not silently baked into `tokenize`.
DEFAULT_STOPWORDS = frozenset(
    """
    lÃƒÂ  cÃ¡Â»Â§a vÃƒÂ  cÃƒÂ¡c nhÃ¡Â»Â¯ng nÃƒÂ y Ã„â€˜ÃƒÂ³ cho Ã„â€˜Ã¡Â»Æ’ vÃ¡Â»â€ºi trong trÃƒÂªn dÃ†Â°Ã¡Â»â€ºi khi Ã„â€˜ÃƒÂ£ sÃ¡ÂºÂ½ cÃƒÂ³ bao nhiÃƒÂªu
    cÃƒÂ´ng ty mÃ¡Â»â„¢t Ã„â€˜Ã†Â°Ã¡Â»Â£c tÃ¡ÂºÂ¡i vÃ¡Â»Â tÃ¡Â»Â« Ã„â€˜Ã¡ÂºÂ¿n theo nhÃ†Â° sau trÃ†Â°Ã¡Â»â€ºc hay hoÃ¡ÂºÂ·c thÃƒÂ¬ mÃƒÂ  nÃƒÂ o gÃƒÂ¬
    nÃ„Æ’m ngÃƒÂ y thÃƒÂ¡ng Ã„â€˜Ã†Â¡n vÃ¡Â»â€¹ tÃƒÂ­nh vnd Ã„â€˜Ã¡Â»â€œng cÃ¡Â»â€¢ phÃ¡ÂºÂ§n
    """.split()
)


def tokenize(text: str, stopwords: frozenset = frozenset()) -> list[str]:
    r"""Lowercase Unicode word tokens. `\w` is Unicode-aware for str input in Python 3, so
    Vietnamese diacritic letters are kept intact (not stripped as "non-word" characters).
    `stopwords` defaults to empty (no filtering) -- callers needing the corpus-tuned default
    pass `stopwords=DEFAULT_STOPWORDS` explicitly; see module docstring above for why this
    isn't silently always-on."""
    tokens = _TOKEN_RE.findall(text.lower())
    return [t for t in tokens if t not in stopwords] if stopwords else tokens


def build_enriched_document_text(
    ticker: str, company_name: str, year, searchable_text: str, identity_boost: int = 5
) -> str:
    """Compose a table's indexed text from its structured identity (ticker/company/year,
    already available from normalization) plus its cell text, repeating the identity fields
    `identity_boost` times.

    Why repetition, not just concatenation once: a table's cell text can run to hundreds of
    tokens (large tables), so a ticker/company name appearing once has negligible term
    frequency next to that volume. Repeating it lets its term frequency compete -- measured
    necessity (see DEFAULT_STOPWORDS docstring): without this, recall_macro on the Checkpoint 2
    dev set was 0.21 (stopwords filtered, no boost); with 5x repetition, 0.46.
    """
    identity = f"{ticker} {company_name} {year} " * identity_boost
    return identity + searchable_text


@dataclass
class BM25Index:
    doc_ids: list  # opaque identifiers, parallel to the corpus passed to build()
    doc_len: list  # token count per document
    doc_freqs: list[dict]  # per-doc term -> count
    postings: dict  # term -> list[doc_idx] (documents containing that term at all)
    df: dict  # term -> number of docs containing it
    avgdl: float
    n_docs: int
    k1: float = 1.5
    b: float = 0.75
    # stored (not re-passed at search time) so a query is always tokenized the same way its
    # documents were indexed -- passing mismatched stopwords between build_index() and search()
    # would silently break every postings lookup.
    stopwords: frozenset = field(default_factory=frozenset)

    def idf(self, term: str) -> float:
        n_t = self.df.get(term, 0)
        # BM25+-style idf (add 1 inside the log) keeps every idf non-negative, which matters
        # here because table "documents" are short (a caption + a handful of cells) and common
        # terms can appear in a large fraction of a 100K+-table corpus.
        return math.log(1.0 + (self.n_docs - n_t + 0.5) / (n_t + 0.5))

    def _score_doc(self, query_tokens: list[str], doc_idx: int) -> float:
        freqs = self.doc_freqs[doc_idx]
        dl = self.doc_len[doc_idx]
        total = 0.0
        for t in query_tokens:
            f = freqs.get(t, 0)
            if f == 0:
                continue
            num = f * (self.k1 + 1)
            den = f + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
            total += self.idf(t) * (num / den)
        return total

    def search(self, query: str, top_k: int | None = None) -> list[tuple]:
        """Returns (doc_id, score) pairs sorted by descending score. If top_k is None, returns
        every document with a nonzero score (no candidate silently dropped) -- which, for the
        inverted-index implementation, is exactly the set of documents sharing at least one
        query term."""
        query_tokens = tokenize(query, stopwords=self.stopwords)
        candidate_doc_idxs: set = set()
        for t in query_tokens:
            candidate_doc_idxs.update(self.postings.get(t, ()))

        scored = [(self.doc_ids[i], self._score_doc(query_tokens, i)) for i in candidate_doc_idxs]
        scored.sort(key=lambda x: (-x[1], str(x[0])))
        return scored[:top_k] if top_k is not None else scored


def build_index(
    doc_ids: list,
    texts: list[str],
    k1: float = 1.5,
    b: float = 0.75,
    stopwords: frozenset = frozenset(),
) -> BM25Index:
    if len(doc_ids) != len(texts):
        raise ValueError("doc_ids and texts must be the same length")

    doc_len = []
    doc_freqs = []
    postings: dict = {}
    df: dict = {}
    total_len = 0

    for doc_idx, text in enumerate(texts):
        tokens = tokenize(text, stopwords=stopwords)
        total_len += len(tokens)
        doc_len.append(len(tokens))
        counts: dict = {}
        for tok in tokens:
            counts[tok] = counts.get(tok, 0) + 1
        doc_freqs.append(counts)
        for tok in counts:
            df[tok] = df.get(tok, 0) + 1
            postings.setdefault(tok, []).append(doc_idx)

    n_docs = len(texts)
    avgdl = (total_len / n_docs) if n_docs else 0.0
    return BM25Index(
        doc_ids=doc_ids,
        doc_len=doc_len,
        doc_freqs=doc_freqs,
        postings=postings,
        df=df,
        avgdl=avgdl,
        n_docs=n_docs,
        k1=k1,
        b=b,
        stopwords=stopwords,
    )


Overwriting src/retrieval/sparse.py


In [73]:
%%writefile eval/__init__.py
# see eval/metrics.py (P/R/F2) and eval/run_eval.py (dev-set evaluation runner).


Overwriting eval/__init__.py


In [74]:
%%writefile eval/metrics.py
"""Retrieval evaluation metrics: Precision, Recall, F2 (macro-averaged), per docs/eval.md.

    Precision = mean over queries of (correct retrieved / retrieved)
    Recall    = mean over queries of (correct retrieved / relevant)
    F2        = 5*P*R / (4*P + R)   -- recall weighted 4:1 over precision (CONTEXT.md Ã‚Â§4)

`docs/eval.md` does not specify whether "correct retrieved table" is judged by document ID,
table ID, or exact `relevant_tables` (report_id|line_position) match -- CONTEXT.md explicitly
flags this as unresolved and says metric code should be adjustable rather than hardcoding an
assumption. `key_fn` on `precision_recall_f2_for_query` is that adjustment point: pass
`key_fn=lambda s: s.split("|")[0]` to compare at report_id (document) granularity instead of
the exact-line default.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Iterable


def _default_key(item: str) -> str:
    return item


@dataclass
class QueryScore:
    query_id: object
    precision: float
    recall: float
    f2: float
    n_retrieved: int
    n_relevant: int
    n_correct: int


def precision_recall_f2_for_query(
    retrieved: Iterable[str],
    relevant: Iterable[str],
    key_fn: Callable[[str], str] = _default_key,
) -> tuple[float, float, float]:
    """Precision/Recall/F2 for one query. Empty `retrieved` -> precision 0 (not undefined) to
    stay macro-averageable; empty `relevant` -> recall 0 by the same reasoning (a dev-set
    construction bug, not something that should silently vanish from the average)."""
    retrieved_keys = {key_fn(r) for r in retrieved}
    relevant_keys = {key_fn(r) for r in relevant}
    n_correct = len(retrieved_keys & relevant_keys)

    precision = n_correct / len(retrieved_keys) if retrieved_keys else 0.0
    recall = n_correct / len(relevant_keys) if relevant_keys else 0.0
    if precision + recall == 0:
        f2 = 0.0
    else:
        f2 = (5 * precision * recall) / (4 * precision + recall)
    return precision, recall, f2


def evaluate_retrieval(
    queries: list[dict],
    retrieved_by_id: dict,
    key_fn: Callable[[str], str] = _default_key,
) -> tuple[list[QueryScore], dict]:
    """queries: list of {"id": ..., "relevant_tables": [...]}.
    retrieved_by_id: {query_id: [retrieved table-key strings, ranked or not]}.
    Returns (per-query scores, macro-averaged summary dict)."""
    scores = []
    for q in queries:
        qid = q["id"]
        relevant = q["relevant_tables"]
        retrieved = retrieved_by_id.get(qid, [])
        p, r, f2 = precision_recall_f2_for_query(retrieved, relevant, key_fn=key_fn)
        retrieved_keys = {key_fn(x) for x in retrieved}
        relevant_keys = {key_fn(x) for x in relevant}
        scores.append(
            QueryScore(
                query_id=qid,
                precision=p,
                recall=r,
                f2=f2,
                n_retrieved=len(retrieved_keys),
                n_relevant=len(relevant_keys),
                n_correct=len(retrieved_keys & relevant_keys),
            )
        )

    n = len(scores)
    summary = {
        "n_queries": n,
        "precision_macro": sum(s.precision for s in scores) / n if n else 0.0,
        "recall_macro": sum(s.recall for s in scores) / n if n else 0.0,
        "f2_macro": sum(s.f2 for s in scores) / n if n else 0.0,
    }
    return scores, summary


Overwriting eval/metrics.py


In [75]:
sys.path.insert(0, ".")  # so `import eval.metrics` resolves at repo root, alongside src/
import retrieval.sparse as rs
import eval.metrics as em
importlib.reload(rs)
importlib.reload(em)

ticker_name = {}
with open(RAW_DIR / "hf_meta" / "code_stock.csv", encoding="utf-8") as f:
    reader = csv.reader(f)
    next(reader)
    for t, name in reader:
        ticker_name[t] = name

norm_df = pd.read_csv(normalized_path, dtype={"year": "Int64"}, keep_default_na=False)
doc_ids = (norm_df["report_id"] + "|" + norm_df["line_position"].astype(str)).tolist()
texts_bare = norm_df["searchable_text"].tolist()
# first attempt: bare cell text, no company/year context, no stopword filtering -- kept as the
# "naive" baseline below specifically to show why it fails, not because it's recommended.
texts_final = [
    rs.build_enriched_document_text(t, ticker_name.get(t, ""), y, s)
    for t, y, s in zip(norm_df["ticker"], norm_df["year"], norm_df["searchable_text"])
]
print(f"{len(doc_ids)} indexed tables")


146243 indexed tables


In [76]:
from pathlib import Path
import shutil

source = Path("/kaggle/input/datasets/tofuwonion/snapshot/dev_questions")
target = Path("/kaggle/working/eval/dev_questions")

assert (source / "dev_v1.jsonl").is_file(), f"KhÃƒÂ´ng thÃ¡ÂºÂ¥y file nguÃ¡Â»â€œn: {source / 'dev_v1.jsonl'}"
shutil.copytree(source, target, dirs_exist_ok=True)

print("Ã„ÂÃƒÂ£ chÃƒÂ©p:", target / "dev_v1.jsonl")

Ã„ÂÃƒÂ£ chÃƒÂ©p: /kaggle/working/eval/dev_questions/dev_v1.jsonl


In [77]:
t0 = time.time()
index_bare = rs.build_index(doc_ids, texts_bare, stopwords=rs.DEFAULT_STOPWORDS)
index_final = rs.build_index(doc_ids, texts_final, stopwords=rs.DEFAULT_STOPWORDS)
print(f"built both BM25 indexes in {time.time()-t0:.0f}s")

dev_questions = []
with open(DEV_QUESTIONS_DIR / "dev_v1.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            dev_questions.append(json.loads(line))
print(f"{len(dev_questions)} dev questions loaded")

built both BM25 indexes in 38s
18 dev questions loaded


In [78]:
def rankings_for(index):
    return {q["id"]: index.search(q["question"], top_k=30) for q in dev_questions}

def eval_at(rankings, top_k):
    retrieved_by_id = {qid: [d for d, _ in ranked[:top_k]] for qid, ranked in rankings.items()}
    return em.evaluate_retrieval(dev_questions, retrieved_by_id)

rankings_bare = rankings_for(index_bare)
rankings_final = rankings_for(index_final)

print("{:45s} {:>6s} {:>6s} {:>6s} {:>6s}".format("variant", "top_k", "P", "R", "F2"))
for label, rankings in [("bare (no company/year context)", rankings_bare), ("final (identity-boosted + stopwords)", rankings_final)]:
    for top_k in (5, 10, 20, 30):
        _, summary = eval_at(rankings, top_k)
        p, r, f2 = summary["precision_macro"], summary["recall_macro"], summary["f2_macro"]
        print(f"{label:45s} {top_k:>6d} {p:>6.3f} {r:>6.3f} {f2:>6.3f}")

variant                                        top_k      P      R     F2
bare (no company/year context)                     5  0.000  0.000  0.000
bare (no company/year context)                    10  0.000  0.000  0.000
bare (no company/year context)                    20  0.000  0.000  0.000
bare (no company/year context)                    30  0.000  0.000  0.000
final (identity-boosted + stopwords)               5  0.089  0.389  0.228
final (identity-boosted + stopwords)              10  0.050  0.444  0.170
final (identity-boosted + stopwords)              20  0.025  0.444  0.101
final (identity-boosted + stopwords)              30  0.019  0.500  0.080


In [79]:
scores, summary = eval_at(rankings_final, 10)
qmap = {q["id"]: q for q in dev_questions}
print(f"per-query detail, final index, top_k=10 (measured on dev_v1.jsonl, {len(dev_questions)} hand-built questions)")
for s in scores:
    q = qmap[s.query_id]
    cat = q["category"]
    print(f"  id={s.query_id:4d} [{cat:25s}] P={s.precision:.2f} R={s.recall:.2f} F2={s.f2:.2f}  correct={s.n_correct}/{s.n_relevant}")

reports_dir = EVAL_DIR / "reports"; reports_dir.mkdir(parents=True, exist_ok=True)
rankings_json = {str(qid): ranked for qid, ranked in rankings_final.items()}
(reports_dir / "retrieval_dev_v1_bm25_rankings.json").write_text(json.dumps(rankings_json, ensure_ascii=False, indent=2), encoding="utf-8")
variants = {}
for top_k in (5, 10, 20, 30):
    per_query, aggregate = eval_at(rankings_final, top_k)
    variants[str(top_k)] = {"summary": aggregate, "per_query": [x.__dict__ for x in per_query]}
report = {"dev_set": "eval/dev_questions/dev_v1.jsonl", "ground_truth_status": "self-constructed diagnostic set; not official labels", "matching": "exact report_id|line_position", "approved_top_k": 10, "n_tables": len(norm_df), "variants": variants}
(reports_dir / "retrieval_dev_v1_bm25_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

per-query detail, final index, top_k=10 (measured on dev_v1.jsonl, 18 hand-built questions)
  id=   1 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id=   2 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id=   4 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id=   5 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id=   6 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id=   7 [simple_lookup            ] P=0.00 R=0.00 F2=0.00  correct=0/1
  id=   9 [simple_lookup            ] P=0.00 R=0.00 F2=0.00  correct=0/1
  id=  13 [simple_lookup            ] P=0.00 R=0.00 F2=0.00  correct=0/1
  id=  25 [simple_lookup            ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id= 584 [multi_year_comparison    ] P=0.00 R=0.00 F2=0.00  correct=0/1
  id= 598 [multi_year_comparison    ] P=0.10 R=1.00 F2=0.36  correct=1/1
  id= 604 [multi_year_comparison    ] P=0.00 R=0.00 F2=0.00  correct=0/1
  id= 405 [derived_ratio        

16395

In [80]:
CHECKPOINT_2_APPROVED = True  # approved by the user in chat, 2026-08-23: BM25 enriched, top_k=10
assert CHECKPOINT_2_APPROVED, (
    "Checkpoint 2 must be explicitly approved before Checkpoint 3."
)

## Checkpoint 3 Ã¢â‚¬â€ Schema Linking + Text-to-Pandas + Execution/Repair *(KAGGLE RUN REQUIRED)*

Approved pipeline component: `Qwen/Qwen2.5-Coder-7B-Instruct-AWQ` (Apache-2.0, 7.61B,
released 2024-09-19). Weights are an attached Kaggle Dataset and inference is offline.
The model selects only allow-listed row/column operands; validation rejects invented schema,
then deterministic code renders the Pandas query. Every generate/validate/execute/repair
attempt is persisted. Local answer tolerance is `abs_tol=rel_tol=1e-6`, explicitly a
development assumption because BTC has not published its tolerance.


In [82]:
from pathlib import Path

matches = list(Path("/kaggle/input").glob("**/src-code/query_generation/generator.py"))
print(matches)

assert matches, "ChÃ†Â°a attach Dataset chÃ¡Â»Â©a source code ViFinQA"
CODE_ROOT = matches[0].parents[2]
print("CODE_ROOT =", CODE_ROOT)

[PosixPath('/kaggle/input/datasets/tofuwonion/snapshot/src-code/query_generation/generator.py')]
CODE_ROOT = /kaggle/input/datasets/tofuwonion/snapshot


In [83]:
import importlib.util
import json
import os
import sys

CODE_ROOT = Path("/kaggle/input/datasets/tofuwonion/snapshot")
SRC_ROOT = CODE_ROOT / "src-code"
EVAL_SOURCE_ROOT = CODE_ROOT / "eval-code"

MODEL_PATH = Path(os.environ.get(
  "VIFINQA_QWEN_MODEL_PATH",
  "/kaggle/input/datasets/tofuwonion/qwen25-coder-7b-instruct-awq/qwen25-coder-7b-instruct-awq",
))

assert (SRC_ROOT / "query_generation" / "generator.py").exists(), (
  f"KhÃƒÂ´ng tÃƒÂ¬m thÃ¡ÂºÂ¥y generator.py: {SRC_ROOT}"
)
assert (SRC_ROOT / "common" / "table_store.py").exists(), (
  f"Snapshot thiÃ¡ÂºÂ¿u common/table_store.py: {SRC_ROOT}"
)
assert (SRC_ROOT / "pipeline.py").exists(), (
  f"Snapshot thiÃ¡ÂºÂ¿u pipeline.py: {SRC_ROOT}"
)
assert (EVAL_SOURCE_ROOT / "__init__.py").exists(), (
  f"eval-src thiÃ¡ÂºÂ¿u __init__.py: {EVAL_SOURCE_ROOT}"
)
assert (EVAL_SOURCE_ROOT / "run_generation_eval.py").exists(), (
  f"KhÃƒÂ´ng tÃƒÂ¬m thÃ¡ÂºÂ¥y run_generation_eval.py: {EVAL_SOURCE_ROOT}"
)
assert MODEL_PATH.exists(), f"ChÃ†Â°a attach Qwen AWQ Dataset: {MODEL_PATH}"

# BÃ¡ÂºÂ£o Ã„â€˜Ã¡ÂºÂ£m code artifact Ã„â€˜Ã†Â°Ã¡Â»Â£c Ã†Â°u tiÃƒÂªn hÃ†Â¡n cÃƒÂ¡c module Ã„â€˜ÃƒÂ£ cÃƒÂ³ trong kernel.
sys.path = [
  str(SRC_ROOT),
  str(CODE_ROOT),
  *[p for p in sys.path if p not in {str(SRC_ROOT), str(CODE_ROOT)}],
]

_snapshot_package_names = (
  "common", "execution", "extraction", "normalization",
  "pipeline", "query_generation", "retrieval", "schema_linking", "eval",
)
for module_name in list(sys.modules):
  if module_name in _snapshot_package_names or module_name.startswith(
      tuple(f"{name}." for name in _snapshot_package_names)
  ):
      del sys.modules[module_name]

# `eval-src` cÃƒÂ³ dÃ¡ÂºÂ¥u "-", nÃƒÂªn ÃƒÂ¡nh xÃ¡ÂºÂ¡ nÃƒÂ³ thÃƒÂ nh package Python tÃƒÂªn `eval`.
spec = importlib.util.spec_from_file_location(
  "eval",
  EVAL_SOURCE_ROOT / "__init__.py",
  submodule_search_locations=[str(EVAL_SOURCE_ROOT)],
)
assert spec is not None and spec.loader is not None, "KhÃƒÂ´ng thÃ¡Â»Æ’ nÃ¡ÂºÂ¡p eval-src"

eval_package = importlib.util.module_from_spec(spec)
sys.modules["eval"] = eval_package
spec.loader.exec_module(eval_package)

_full_corpus_path = SRC_ROOT / 'retrieval' / 'full_corpus.py'
if not _full_corpus_path.exists():
    _full_retrieval = type(sys)('retrieval.full_corpus')
    _full_retrieval.TICKER_TOKEN_RE = __import__('re').compile(r'\b(?=[A-Z0-9]*[A-Z])[A-Z0-9]{2,5}\b')
    sys.modules['retrieval.full_corpus'] = _full_retrieval

from dataclasses import dataclass
import json as _json
import re as _re
import traceback
import unicodedata as _unicodedata

import execution.runner as _runner
import pipeline as _pipeline
import query_generation.generator as _generator
from query_generation.generator import BoundOperand, QueryPlan
from schema_linking.linker import SchemaLinkResult

PATCH_REVISION = 'constraint-first-v6'

class PromptBudgetError(RuntimeError):
    stage = 'prompt_budget'

def _compact_prompt_payload(linked):
    return {'question': linked.question, 'query_family': linked.query_family,
            'requested_unit': linked.requested_unit, 'allowed_operands': [
                {'table_key': x.table_key, 'row_index': x.row_index,
                 'column_index': x.column_index, 'row_label': x.row_label,
                 'column_header': x.column_header, 'raw_value': x.raw_value}
                for x in linked.operands]}

SchemaLinkResult.prompt_payload = _compact_prompt_payload

def _model_input_device(generator):
    try:
        return generator.model.get_input_embeddings().weight.device
    except AttributeError:
        return next(generator.model.parameters()).device

def _tokenized_inputs(generator, prompt, *, move_to_model=False):
    inputs = generator.tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}], add_generation_prompt=True,
        tokenize=True, return_tensors='pt')
    return inputs.to(_model_input_device(generator)) if move_to_model else inputs

def _context_limit(generator):
    values = [getattr(generator.model.config, 'max_position_embeddings', None),
              getattr(generator.tokenizer, 'model_max_length', None)]
    valid = [int(x) for x in values if isinstance(x, int) and 0 < x < 1_000_000]
    if not valid:
        raise PromptBudgetError('model exposes no finite context limit')
    return min(valid)

# SDPA prefill grows quadratically with input tokens.  The coefficient below
# is deliberately conservative: it was estimated from the persisted 7.6--9.5
# GiB OOM allocations for 8.5k--9.4k-token prompts on the Kaggle GPU.
_ATTENTION_BYTES_PER_TOKEN_SQUARED = 110
_VRAM_RESERVE_BYTES = 1 << 30
_DEFAULT_MAX_INPUT_TOKENS = 3500

def _vram_input_budget(generator):
    import torch
    configured = int(os.environ.get('VIFINQA_MAX_INPUT_TOKENS', _DEFAULT_MAX_INPUT_TOKENS))
    try:
        free_bytes, total_bytes = torch.cuda.mem_get_info(_model_input_device(generator))
    except (AttributeError, RuntimeError):
        return configured, None
    usable_bytes = max(0, free_bytes - _VRAM_RESERVE_BYTES)
    quadratic_limit = int((usable_bytes / _ATTENTION_BYTES_PER_TOKEN_SQUARED) ** 0.5)
    # At least 1k tokens is required to retain a meaningful grounded plan; fail
    # explicitly instead of silently dropping every operand below that threshold.
    if quadratic_limit < 1024:
        raise PromptBudgetError(f'insufficient free VRAM: {free_bytes} bytes')
    return min(configured, quadratic_limit), {'free_vram_bytes': free_bytes,
        'total_vram_bytes': total_bytes, 'vram_reserve_bytes': _VRAM_RESERVE_BYTES,
        'quadratic_token_limit': quadratic_limit}

def _fit_prompt(generator, prompt):
    prefix, tail = prompt.split('CONTEXT=', 1)
    context_text, feedback = tail.rsplit('\nREPAIR_FEEDBACK=', 1)
    context = _json.loads(context_text)
    candidates, chosen = context['allowed_operands'], []
    max_new_tokens, margin = 256, 256
    limit = _context_limit(generator)
    vram_limit, vram_diagnostics = _vram_input_budget(generator)
    retry_cap = getattr(generator, '_vifinqa_retry_input_cap', None)
    budget = min(limit - max_new_tokens - margin, vram_limit, retry_cap or vram_limit)
    for candidate in candidates:
        proposal = dict(context, allowed_operands=chosen + [candidate])
        candidate_prompt = prefix + 'CONTEXT=' + _json.dumps(proposal, ensure_ascii=False, separators=(',', ':')) + '\nREPAIR_FEEDBACK=' + feedback
        if int(_tokenized_inputs(generator, candidate_prompt)['input_ids'].shape[-1]) > budget:
            break
        chosen.append(candidate)
    if not chosen:
        raise PromptBudgetError(f'first grounded candidate exceeds input budget {budget}')
    fitted_prompt = prefix + 'CONTEXT=' + _json.dumps(dict(context, allowed_operands=chosen), ensure_ascii=False, separators=(',', ':')) + '\nREPAIR_FEEDBACK=' + feedback
    input_tokens = int(_tokenized_inputs(generator, fitted_prompt)['input_ids'].shape[-1])
    assert input_tokens <= budget, (input_tokens, budget)
    generator._vifinqa_prompt_diagnostics = {'strategy': 'bounded_model_generation',
        'context_limit': limit, 'max_new_tokens': max_new_tokens, 'safety_margin': margin,
        'input_budget': budget, 'input_tokens': input_tokens,
        'candidates_available': len(candidates), 'candidates_exposed': len(chosen),
        'retry_input_cap': retry_cap, 'vram': vram_diagnostics}
    generator._vifinqa_visible_operands = {(x['table_key'], x['row_index'], x['column_index']) for x in chosen}
    return fitted_prompt, max_new_tokens

def _safe_model_call(self, prompt):
    import torch
    prompt, max_new_tokens = _fit_prompt(self, prompt)
    inputs = _tokenized_inputs(self, prompt, move_to_model=True)
    input_tokens = int(inputs['input_ids'].shape[-1])
    with torch.inference_mode():
        output = self.model.generate(**inputs, do_sample=False, max_new_tokens=max_new_tokens, use_cache=True)
    return self.tokenizer.decode(output[0][input_tokens:], skip_special_tokens=True)

def _safe_generate_plan(linked, complete, feedback=None):
    payload = _generator._extract_json(complete(_generator.build_prompt(linked, feedback)))
    visible = getattr(complete, '_vifinqa_visible_operands', None)
    if visible is not None:
        linked = SchemaLinkResult(linked.question, linked.query_family, linked.requested_unit,
            linked.table_keys, [x for x in linked.operands if (x.table_key, x.row_index, x.column_index) in visible])
    operand_keys = ('alias', 'table_key', 'row_index', 'column_index')
    raw_operands = payload.get('operands', [])
    if not isinstance(raw_operands, list) or any(not isinstance(x, dict) or any(k not in x for k in operand_keys) for x in raw_operands):
        raise ValueError('model operands must be objects containing alias, table_key, row_index, column_index')
    payload = {**payload, 'operands': [{k: x[k] for k in operand_keys} for x in raw_operands]}
    return _generator.validate_plan(payload, linked)

def _unit_scale(text):
    value = ''.join(ch for ch in _unicodedata.normalize('NFD', (text or '').lower())
                    if not _unicodedata.combining(ch)).replace('Ä‘', 'd')
    if 'nghin ty' in value:
        return 1_000_000_000_000
    if 'ty' in value:
        return 1_000_000_000
    if 'trieu' in value:
        return 1_000_000
    if 'nghin' in value:
        return 1_000
    if 'vnd' in value or 'dong' in value:
        return 1
    return None

def _source_scale(table, operand):
    # Unit nearest the selected cell wins.  A table can legitimately contain several
    # units, so flattening its whole header can silently select an unrelated one.
    column = next((item for item in table.get('column_metadata', [])
                   if item.get('column_index') == operand.column_index), {})
    if column.get('scale_to_vnd') is not None:
        return column['scale_to_vnd']
    sources = [operand.column_header]
    detected = table.get('detected_units', [])
    sources.append(detected if isinstance(detected, str) else ' '.join(detected))
    sources.append(' '.join(str(row[operand.column_index]) for row in table.get('grid', [])[:6]
                            if operand.column_index < len(row)))
    sources.append(' '.join(table.get('caption_context', [])))
    for source in sources:
        scale = _unit_scale(source)
        if scale is not None:
            return scale
    return None

def _fold_text(value):
    return ''.join(ch for ch in _unicodedata.normalize('NFD', (value or '').lower())
                   if not _unicodedata.combining(ch)).replace('đ', 'd')

def _matches_direct_period(question, operand, table):
    q, header, label = _fold_text(question), _fold_text(operand.column_header), _fold_text(operand.row_label)
    requested_years = {int(year) for year in _re.findall(r'\b20\d{2}\b', q)}
    header_years = {int(year) for year in _re.findall(r'\b20\d{2}\b', header)}
    table_year = table.get('year')
    if requested_years:
        if header_years and not (header_years & requested_years):
            return False
        if not header_years and table_year is not None:
            current = 'nam nay' in header
            prior = 'nam truoc' in header or 'dau nam' in label
            implied_year = int(table_year) - 1 if prior and not current else int(table_year)
            if implied_year not in requested_years:
                return False
    if 'cuoi nam' in q and 'dau nam' in label:
        return False
    if 'dau nam' in q and 'cuoi nam' in label:
        return False
    if 'trong nam' in q and any(marker in label for marker in ('dau nam', 'cuoi nam')):
        return False
    return True

def _semantic_metric_score(question, row_label):
    ignored = {'cong', 'ty', 'co', 'phan', 'nam', 'bao', 'nhieu', 'dong', 'trieu', 'ty', 'cua', 'va', 'voi', 'tai'}
    q_terms = {x for x in _re.findall(r'\w+', _fold_text(question)) if len(x) > 3 and x not in ignored and not x.isdigit()}
    label_terms = set(_re.findall(r'\w+', _fold_text(row_label)))
    return len(q_terms & label_terms)

def _direct_lookup_plan(linked, tables_by_key):
    question = linked.question.lower()
    question_years = set(_re.findall(r'\b20\d{2}\b', question))
    candidates = list(linked.operands)
    # A literal ticker is a hard constraint only if it occurs among retrieved tables.
    # This avoids mistaking FPT in the legal name FPTS for a listed-company ticker.
    question_codes = set(_re.findall(r'\b[A-Z]{2,5}\b', linked.question)) - {'BCTC', 'CTCP', 'TMCP', 'TCT', 'TNHH', 'VND'}
    available_tickers = {str(tables_by_key[key].get('ticker', '')) for key in linked.table_keys}
    mentioned_tickers = question_codes & available_tickers
    if mentioned_tickers:
        candidates = [x for x in candidates if tables_by_key[x.table_key].get('ticker') in mentioned_tickers]
        if not candidates:
            raise ValueError('schema linker retained no operands for the explicitly named ticker')
    if 'cÃ´ng ty máº¹' in question or 'cÃ´ng ty riÃªng' in question:
        candidates = [x for x in candidates if tables_by_key[x.table_key].get('variant') == 'separate']
    elif 'há»£p nháº¥t' in question or 'táº­p Ä‘oÃ n' in question:
        candidates = [x for x in candidates if tables_by_key[x.table_key].get('variant') == 'consolidated']
    candidates = [x for x in candidates if _matches_direct_period(linked.question, x, tables_by_key[x.table_key])]
    # A direct answer must reference a numeric data column, never the label/code/header row.
    candidates = [x for x in candidates if x.row_index > 0 and x.column_index > 0
                  and x.column_header.strip()
                  and not any(token in x.column_header.lower() for token in ('mÃ£ sá»‘', 'thuyáº¿t minh', 'note', 'biáº¿n Ä‘á»™ng', '%'))]
    if not candidates:
        raise ValueError('no schema-linked numeric data operand satisfies company/variant constraints')
    def score(x):
        header, row_label = x.column_header.lower(), x.row_label.lower()
        table_year = str(tables_by_key[x.table_key].get('year', ''))
        year_score = 20 if any(year in header for year in question_years) else 0
        current_year_score = 10 if question_years and table_year in question_years and 'nÄƒm nay' in header else 0
        identity = ' '.join(str(tables_by_key[x.table_key].get(key, '')) for key in ('table_identity', 'caption_context')).lower()
        narrative_penalty = 25 if any(token in identity for token in ('giáº£i trÃ¬nh', 'thuyáº¿t minh', 'explan')) else 0
        period_score = 30 if ('cuá»‘i nÄƒm' in question and 'cuá»‘i nÄƒm' in row_label) or ('Ä‘áº§u nÄƒm' in question and 'Ä‘áº§u nÄƒm' in row_label) else 0
        semantic_score = _semantic_metric_score(linked.question, x.row_label) * 40
        return (x.relevance_score * 100) + year_score + current_year_score + period_score + semantic_score - narrative_penalty
    operand = max(candidates, key=lambda x: (score(x), x.table_key, -x.row_index, -x.column_index))
    table = tables_by_key[operand.table_key]
    source_scale = _source_scale(table, operand) or 1
    target_scale = _unit_scale(linked.requested_unit) or source_scale
    factor = source_scale / target_scale
    expression = 'x' if factor == 1 else f'x * {factor!r}'
    return QueryPlan([BoundOperand('x', operand.table_key, operand.row_index, operand.column_index)],
                     expression, linked.requested_unit, 'deterministic direct lookup with variant and unit constraints')

@dataclass
class _SafeExecutionAttempt:
    attempt: int
    pandas_query: str | None
    error_type: str | None
    error_message: str | None
    stage: str
    traceback: str | None = None
    prompt_diagnostics: dict | None = None

def _is_oom(exc):
    return type(exc).__name__ == 'OutOfMemoryError' or 'out of memory' in str(exc).lower()

def _safe_execute_with_repair(linked, tables_by_key, complete, max_retries=2):
    attempts, feedback, query, oom_retried = [], None, None, False
    retries = 0 if linked.query_family == 'direct_lookup' else max_retries
    if linked.query_family != 'direct_lookup':
        complete._vifinqa_retry_input_cap = None
    for attempt_no in range(retries + 1):
        stage = 'generation_or_validation'
        try:
            plan = _direct_lookup_plan(linked, tables_by_key) if linked.query_family == 'direct_lookup' else _safe_generate_plan(linked, complete, feedback)
            stage = 'render'
            frames, variables = _generator.build_evidence_frames(plan, tables_by_key)
            query = _generator.render_pandas_query(plan, variables)
            stage = 'execution'
            answer = _runner.execute_query(query, frames)
            diagnostics = {'strategy': PATCH_REVISION} if linked.query_family == 'direct_lookup' else getattr(complete, '_vifinqa_prompt_diagnostics', None)
            attempts.append(_SafeExecutionAttempt(attempt_no, query, None, None, stage, prompt_diagnostics=diagnostics))
            return _runner.ExecutionResult(True, answer, query, frames, attempts, plan)
        except Exception as exc:
            stage = getattr(exc, 'stage', stage)
            attempts.append(_SafeExecutionAttempt(attempt_no, query, type(exc).__name__, str(exc), stage, traceback.format_exc(), getattr(complete, '_vifinqa_prompt_diagnostics', None)))
            if stage == 'prompt_budget':
                break
            if _is_oom(exc):
                # One retry is useful only with a materially smaller prompt.
                # Never feed the same OOM back as textual repair feedback.
                previous = (getattr(complete, '_vifinqa_prompt_diagnostics', None) or {}).get('input_tokens')
                if oom_retried or not previous:
                    break
                complete._vifinqa_retry_input_cap = max(1024, int(previous * 0.7))
                oom_retried = True
                continue
            feedback = f'{type(exc).__name__}: {exc}'
    return _runner.ExecutionResult(False, None, query, {}, attempts, None)

_generator.QwenAWQGenerator.__call__ = _safe_model_call
_generator.generate_plan = _safe_generate_plan
_runner.generate_plan = _safe_generate_plan
_runner.execute_with_repair = _safe_execute_with_repair
_pipeline.execute_with_repair = _safe_execute_with_repair

# Entity-scoped BM25: resolve a distinctive company name before ranking tables.
# This keeps FTS ("Chá»©ng khoÃ¡n FPT") distinct from ticker FPT and prevents a
# lexically similar table from another company consuming the candidate budget.
import pandas as _pd
from retrieval.sparse import DEFAULT_STOPWORDS as _STOPWORDS, build_enriched_document_text as _enriched, build_index as _build_index, tokenize as _tokenize

def _entity_tickers(question, company_by_ticker):
    explicit = {token for token in _re.findall(r'\b[A-Z]{2,5}\b', question)
                if token not in {'BCTC', 'CTCP', 'TMCP', 'TCT', 'TNHH', 'VND'} and token in company_by_ticker}
    # A literal ticker is stronger evidence than a fuzzy legal-name overlap; this
    # prevents VSC/HNG/DLG questions from being scoped to another issuer.
    if explicit:
        return explicit
    q_tokens = set(_tokenize(question, stopwords=_STOPWORDS))
    noise = {'cÃ´ng', 'ty', 'cá»•', 'pháº§n', 'tá»•ng', 'ngÃ¢n', 'hÃ ng'}
    name_scores = {}
    for ticker, name in company_by_ticker.items():
        overlap = len(q_tokens & (set(_tokenize(name, stopwords=_STOPWORDS)) - noise))
        if overlap >= 2:
            name_scores[ticker] = overlap
    if name_scores:
        best = max(name_scores.values())
        return {ticker for ticker, score in name_scores.items() if score == best}
    return set()

def _entity_scoped_rank_questions(questions, catalog_path, companies_path, output_path, *, top_k=10, row_label_index_path=None):
    tables = _pd.read_csv(catalog_path, usecols=['report_id', 'line_position', 'ticker', 'year', 'searchable_text'], keep_default_na=False)
    companies = _pd.read_csv(companies_path, keep_default_na=False)
    company_by_ticker = dict(zip(companies.iloc[:, 0].astype(str), companies.iloc[:, 1].astype(str)))
    doc_ids = (tables['report_id'] + '|' + tables['line_position'].astype(str)).tolist()
    texts = [_enriched(str(row.ticker), company_by_ticker.get(str(row.ticker), ''), row.year, str(row.searchable_text), identity_boost=5) for row in tables.itertuples()]
    global_index = _build_index(doc_ids, texts, stopwords=_STOPWORDS)
    positions_by_ticker = {}
    for position, ticker in enumerate(tables['ticker'].astype(str)):
        positions_by_ticker.setdefault(ticker, []).append(position)
    scoped_indexes, rankings = {}, {}
    for question in questions:
        scope = tuple(sorted(_entity_tickers(question['question'], company_by_ticker)))
        if scope:
            positions = [position for ticker in scope for position in positions_by_ticker.get(ticker, [])]
            if positions:
                if scope not in scoped_indexes:
                    scoped_indexes[scope] = _build_index([doc_ids[i] for i in positions], [texts[i] for i in positions], stopwords=_STOPWORDS)
                hits = scoped_indexes[scope].search(question['question'], top_k=top_k)
            else:
                # An explicitly resolved issuer with no indexed tables is a
                # diagnosable retrieval miss, never permission to answer from
                # another company.
                hits = []
        else:
            hits = global_index.search(question['question'], top_k=top_k)
        rankings[str(question['id'])] = [[key, score] for key, score in hits]
    output_path = Path(output_path); output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + '.tmp')
    temporary.write_text(_json.dumps(rankings, ensure_ascii=False, indent=2), encoding='utf-8')
    temporary.replace(output_path)
    return rankings

# Older Kaggle snapshot datasets omit retrieval/full_corpus.py. Complete the
# pre-import compatibility module with the API later stages need, preserving
# TICKER_TOKEN_RE that generator/schema_linking imported above.
sys.modules['retrieval.full_corpus'].rank_questions = _entity_scoped_rank_questions

# Prefer the tested source implementation when the snapshot contains it.  An
# older snapshot keeps the compatible entity-scoped implementation above.
if _full_corpus_path.exists():
    _source_retrieval_spec = importlib.util.spec_from_file_location(
        '_vifinqa_source_full_retrieval', _full_corpus_path)
    _source_retrieval = importlib.util.module_from_spec(_source_retrieval_spec)
    _source_retrieval_spec.loader.exec_module(_source_retrieval)
    _entity_scoped_rank_questions = _source_retrieval.rank_questions
    _direct_lookup_plan = _generator.generate_direct_lookup_plan

from eval.run_generation_eval import run as run_generation_eval

# Predictions from the previous overlay are not comparable: direct lookup used the model
# to choose report variants and did not perform unit conversion.  Archive every old record
# once so the resumable evaluator regenerates this dev run under PATCH_REVISION.
_prediction_dir = EVAL_DIR / 'reports' / 'generation_dev_v1' / 'predictions'
_archive_dir = EVAL_DIR / 'reports' / 'generation_dev_v1' / 'pre_constraint_first_v6'
if not _archive_dir.exists():
    for _path in _prediction_dir.glob('*.json') if _prediction_dir.exists() else []:
        _archive_dir.mkdir(parents=True, exist_ok=True)
        _path.replace(_archive_dir / _path.name)

# Row-label reranking sidecar (CHANGE_LOG.md 2026-08-31 row-label-rerank entry): build it here
# too (idempotent, same artifact_exists guard as every other cell) so THIS dev-eval comparison
# reflects the same retrieval this kernel routes into the actual Checkpoint-4 submission, not a
# strictly weaker no-rerank baseline.
from retrieval.rerank import build_row_label_index as _build_row_label_index
row_label_index_path = PROCESSED_DIR / "row_label_index.csv"
if not artifact_exists(row_label_index_path):
    _build_row_label_index(structured_path, row_label_index_path)

_entity_rankings_path = EVAL_DIR / 'reports' / 'retrieval_dev_v1_entity_scoped_rankings.json'
_entity_scoped_rank_questions([_json.loads(line) for line in (DEV_QUESTIONS_DIR / 'dev_v1.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()], normalized_path, RAW_DIR / 'hf_meta/code_stock.csv', _entity_rankings_path, top_k=10, row_label_index_path=row_label_index_path)
generation_report = run_generation_eval(
  MODEL_PATH,
  structured_path,
  _entity_rankings_path,
  DEV_QUESTIONS_DIR / "dev_v1.jsonl",
  DEV_QUESTIONS_DIR / "dev_v1_answers.jsonl",
  EVAL_DIR / "reports/generation_dev_v1",
)

print(json.dumps(generation_report, ensure_ascii=False, indent=2))

# Model attempts must satisfy the tokenizer-derived input budget.  Direct lookups do not
# call the model and instead record PATCH_REVISION as their execution strategy.
prediction_dir = EVAL_DIR / 'reports' / 'generation_dev_v1' / 'predictions'
prediction_records = [
    json.loads(path.read_text(encoding='utf-8'))
    for path in sorted(prediction_dir.glob('*.json'))
]
model_attempts = [
    attempt for record in prediction_records for attempt in record.get('attempts', [])
    if (attempt.get('prompt_diagnostics') or {}).get('strategy') == 'bounded_model_generation'
]
budgeted_model_attempts = [
    attempt for attempt in model_attempts
    if {'input_tokens', 'input_budget', 'candidates_exposed'} <= set(attempt.get('prompt_diagnostics') or {})
]
assert len(prediction_records) == len(generation_report['per_query']), 'missing generation records'
assert model_attempts, 'no tokenizer budget diagnostics were persisted for non-direct queries'
assert budgeted_model_attempts, 'no complete tokenizer budget diagnostics were persisted'
assert all(
    x['prompt_diagnostics']['input_tokens'] <= x['prompt_diagnostics']['input_budget']
    for x in budgeted_model_attempts
), 'a generation attempt exceeded its input budget'
assert not any(x.get('stage') == 'prompt_budget' for x in model_attempts), (
    'at least one question has no grounded candidate that fits the context budget'
)
print(json.dumps({
    'model_generation_attempts': len(model_attempts),
    'budgeted_model_generation_attempts': len(budgeted_model_attempts),
    'max_input_tokens': max(x['prompt_diagnostics']['input_tokens'] for x in budgeted_model_attempts),
    'min_candidates_exposed': min(x['prompt_diagnostics']['candidates_exposed'] for x in budgeted_model_attempts),
}, ensure_ascii=False, indent=2))

# These manual dev_v1 labels are diagnostics, not official ground truth.  Keep the
# known cases visible, but do not make their answer accuracy a runtime precondition for
# producing a submission artifact.
records_by_id = {record['id']: record for record in prediction_records}
scores_by_id = {score['id']: score for score in generation_report['per_query']}
assert records_by_id.keys() == scores_by_id.keys(), (
    'prediction records and evaluation scores must cover the same question IDs'
)
regression_diagnostics = [
    {
        'id': question_id,
        'executed': records_by_id[question_id]['executed'],
        'answer': records_by_id[question_id]['answer'],
        'expected_answer': records_by_id[question_id]['expected_answer'],
        'answer_correct': scores_by_id[question_id]['answer_correct'],
        'strategy': (records_by_id[question_id]['attempts'][0].get('prompt_diagnostics') or {}).get('strategy'),
    }
    for question_id in (1, 2, 13)
]
print(json.dumps({'manual_regression_diagnostics': regression_diagnostics}, ensure_ascii=False, indent=2))

PROCEED_TO_MVP_CHECKPOINT_4 = True
assert PROCEED_TO_MVP_CHECKPOINT_4

{
  "n_queries": 18,
  "abs_tolerance_assumption": 1e-06,
  "rel_tolerance_assumption": 1e-06,
  "execution_accuracy": 0.3888888888888889,
  "answer_accuracy": 0.3888888888888889,
  "by_query_family": {
    "comparison": {
      "n": 4,
      "execution_accuracy": 0.0,
      "answer_accuracy": 0.0
    },
    "cross_company_comparison": {
      "n": 1,
      "execution_accuracy": 0.0,
      "answer_accuracy": 0.0
    },
    "direct_lookup": {
      "n": 11,
      "execution_accuracy": 0.6363636363636364,
      "answer_accuracy": 0.6363636363636364
    },
    "growth": {
      "n": 1,
      "execution_accuracy": 0.0,
      "answer_accuracy": 0.0
    },
    "ratio_or_derived": {
      "n": 1,
      "execution_accuracy": 0.0,
      "answer_accuracy": 0.0
    }
  },
  "per_query": [
    {
      "id": 1,
      "query_family": "direct_lookup",
      "executed": true,
      "answer_correct": true,
      "execution_correct": true
    },
    {
      "id": 2,
      "query_family": "direct_lookup"

In [84]:
import json

import pandas as pd
import torch

from eval.metrics import evaluate_retrieval
from eval.run_dense_retrieval import run as run_dense_retrieval
from retrieval.hybrid import fuse_ranking_artifacts

assert torch.cuda.is_available(), "A Kaggle GPU accelerator is required for this step."
print("Dense device:", torch.cuda.get_device_name(0),
      "| VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

# 1) Dense now uses the same enriched representation as BM25 (ticker+company+year
_dense_catalog_columns = set(pd.read_csv(normalized_path, nrows=0).columns)
_dense_required_columns = {"ticker", "year", "searchable_text"}
assert _dense_required_columns <= _dense_catalog_columns, (
    f"normalized_path is missing {_dense_required_columns - _dense_catalog_columns} -- re-run the "
    "Checkpoint 2 section 2.1 normalization cell (writes normalized_tables_entity_v6.csv) before "
    "this dense step"
)

# 2) Build (or reuse) the FULL-corpus dense index and rank dev_v1 through it -- both via the
#    shared, tested retrieval.dense.rank_questions_dense entry point.
# Renamed from dense_bge_m3_full_corpus: enriched-text embeddings are not compatible with a
# cached schema_only index under the old name (run_dense_retrieval only checks doc_ids match,
# not which text built them -- a stale cache under the old name would be silently reused).
DENSE_INDEX_DIR = EVAL_DIR / "reports" / "dense_bge_m3_enriched_full_corpus"
DENSE_RANKINGS_PATH = EVAL_DIR / "reports" / "retrieval_dev_v1_dense_rankings.json"
run_dense_retrieval(
    None, normalized_path, RAW_DIR / "hf_meta/code_stock.csv",
    DEV_QUESTIONS_DIR / "dev_v1.jsonl", DENSE_INDEX_DIR, DENSE_RANKINGS_PATH,
    top_k=10, hf_repo_id="BAAI/bge-m3",
)

# 3) Fuse with the existing BM25 ranking artifact and compare against the CHANGE_LOG acceptance
#    rule: no regression vs the BM25 baseline / already-1.0-recall simple_lookup queries, material
#    improvement on the Q405/Q584/Q591/Q627/Q961-style misses.
HYBRID_RANKINGS_PATH = EVAL_DIR / "reports" / "retrieval_dev_v1_hybrid_rankings.json"
hybrid_rankings = fuse_ranking_artifacts(
    _entity_rankings_path, DENSE_RANKINGS_PATH, HYBRID_RANKINGS_PATH, top_k=10,
)

_dense_questions = [json.loads(line) for line in (DEV_QUESTIONS_DIR / "dev_v1.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
_bm25_retrieved = {q["id"]: [key for key, _ in json.loads(_entity_rankings_path.read_text(encoding="utf-8"))[str(q["id"])][:10]] for q in _dense_questions}
_hybrid_retrieved = {q["id"]: [key for key, _ in hybrid_rankings[str(q["id"])][:10]] for q in _dense_questions}
_bm25_scores, _bm25_summary = evaluate_retrieval(_dense_questions, _bm25_retrieved)
_hybrid_scores, _hybrid_summary = evaluate_retrieval(_dense_questions, _hybrid_retrieved)
print(json.dumps({
    "acceptance_rule": "no regression vs bm25_top10 on simple_lookup / overall; material gain on 405/584/591/627/961-style misses",
    "bm25_top10": _bm25_summary,
    "hybrid_top10": _hybrid_summary,
    "per_question_recall_delta": {
        str(q["id"]): _hybrid_summary and (
            [s for s in _hybrid_scores if s.query_id == q["id"]][0].recall - [s for s in _bm25_scores if s.query_id == q["id"]][0].recall
        )
        for q in _dense_questions
    },
}, ensure_ascii=False, indent=2))

# Do NOT auto-route into submission/run_full_inference.py here -- only do that after eyeballing
# the acceptance-rule comparison above. See Checkpoint 4 for the opt-in --dense-hf-repo-id /
# --dense-index-dir submission invocation.

Dense device: Tesla T4 | VRAM GiB: 14.6


INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-m3/revision/main "HTTP/1.1 200 OK"


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

INFO:sentence_transformers.base.model:Loading SentenceTransformer model from /root/.cache/huggingface/hub/models--BAAI--bge-m3/snapshots/5617a9f61b028005a4858fdac845db406aefb181.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

{
  "acceptance_rule": "no regression vs bm25_top10 on simple_lookup / overall; material gain on 405/584/591/627/961-style misses",
  "bm25_top10": {
    "n_queries": 18,
    "precision_macro": 0.12222222222222223,
    "recall_macro": 0.8333333333333334,
    "f2_macro": 0.3544295210961878
  },
  "hybrid_top10": {
    "n_queries": 18,
    "precision_macro": 0.12222222222222223,
    "recall_macro": 0.8333333333333334,
    "f2_macro": 0.3544295210961878
  },
  "per_question_recall_delta": {
    "1": 0.0,
    "2": 0.0,
    "4": 0.0,
    "5": 0.0,
    "6": 0.0,
    "7": 0.0,
    "9": 0.0,
    "13": 0.0,
    "25": 0.0,
    "584": 0.0,
    "598": 0.0,
    "604": 0.0,
    "405": 0.0,
    "795": 0.0,
    "627": 0.0,
    "591": 0.0,
    "959": 0.0,
    "961": 0.0
  }
}


## Checkpoint 4 Ã¢â‚¬â€ Full inference + validated MVP submission *(KAGGLE RUN)*

Implemented as a resumable Kaggle stage. It reuses valid retrieval/prediction artifacts and
writes evidence CSVs under the package's `data/` directory. Failures stay explicit under
`work/failures`; the formatter refuses missing answers. The validator checks exact official
coverage, schema and paths, then reloads every CSV and executes every pandas query.


In [88]:
import importlib
import sys

# Checkpoint 3 installs this runtime module for snapshots that predate full_corpus.py.
assert "retrieval.full_corpus" in sys.modules, (
  "Run Checkpoint 3 in this kernel before Checkpoint 4; its retrieval runtime overlay is required."
)
assert (CODE_ROOT / "submission" / "run_full_inference.py").exists(), (
  f"Snapshot thiÃ¡ÂºÂ¿u submission/run_full_inference.py: {CODE_ROOT}"
)

# Keep the Checkpoint-3 runtime overlay alive: clearing project modules here would
# silently restore the read-only snapshot and make full inference use the old policy.
sys.path = [
  str(SRC_ROOT),
  str(CODE_ROOT),
  *[p for p in sys.path if p not in {str(SRC_ROOT), str(CODE_ROOT)}],
]

assert _pipeline.execute_with_repair is _safe_execute_with_repair, (
  'Run Checkpoint 3 in this kernel before Checkpoint 4; its active safety overlay is required.'
)

from submission.run_full_inference import run as run_full_inference

ModuleNotFoundError: No module named 'run_dense_retrieval'

In [ ]:
from retrieval.rerank import build_row_label_index

# Row-label reranking sidecar (CHANGE_LOG.md 2026-08-31 row-label-rerank entry) is a local-only
# artifact until built here -- Kaggle's normalized_tables_entity_v6.jsonl has never had this
# derived. submission.run_full_inference.run() auto-detects it next to normalized_path when
# present and silently skips reranking when it is not, so this cell is what actually turns that
# auto-detection on for this Kaggle run. Idempotent/resumable like every other artifact_exists
# cell in this notebook.
row_label_index_path = PROCESSED_DIR / "row_label_index.csv"
if artifact_exists(row_label_index_path):
    print("skip (cached):", row_label_index_path)
else:
    n = build_row_label_index(structured_path, row_label_index_path)
    print(f"built row_label_index.csv: {n} rows -> {row_label_index_path}")


In [ ]:
from submission.run_full_inference import run as run_full_inference
working_root = Path('/kaggle/working') if Path('/kaggle/working').exists() else ROOT
SUBMISSION_PACKAGE_DIR = Path(os.environ.get('VIFINQA_SUBMISSION_PACKAGE', working_root / 'vifinqa_submission_package'))
_full_work_dir = SUBMISSION_PACKAGE_DIR / 'work'
_full_revision_marker = _full_work_dir / f'{PATCH_REVISION}.started'
_full_prediction_dir = _full_work_dir / 'predictions'
if not _full_revision_marker.exists():
    # Completed predictions under the prior policy are invalid evidence for this revision.
    # Move them once; later interrupted runs resume the new revision normally.
    if _full_prediction_dir.exists():
        _full_archive_dir = _full_work_dir / 'pre_entity_scoped_column_unit_v5_predictions'
        assert not _full_archive_dir.exists(), _full_archive_dir
        _full_prediction_dir.replace(_full_archive_dir)
    _full_work_dir.mkdir(parents=True, exist_ok=True)
    _full_revision_marker.write_text(PATCH_REVISION + '\n', encoding='utf-8')
# Hybrid (BM25 + dense BGE-M3) is the default retrieval for this submission run --
# backfill_fusion (CHANGE_LOG.md 2026-08-31 hybrid-fusion-backfill entry) confirmed on a real
# Kaggle run to floor at bm25_top10 (never worse) and only add coverage dense uniquely finds, so
# there is no safety reason left to stay BM25-only. dense_index_dir reuses the embedding already
# built/cached by the Checkpoint-3 dense cell above (must run before this cell, which it does in
# a top-to-bottom Run All) -- if that directory is missing, run_dense_retrieval will build it
# from scratch here instead (slower, but self-healing on a clean working directory).
#
# CAUTION if resuming a working directory that already has *complete* predictions from an
# earlier BM25-only Checkpoint 4 run: _is_complete_prediction() only checks answer completeness,
# not which retrieval produced the candidate set, so already-complete predictions are reused
# as-is and will NOT be recomputed under hybrid. Clear SUBMISSION_PACKAGE_DIR / 'work/predictions'
# first if you need every prediction regenerated under hybrid specifically.
full_report = run_full_inference(
    MODEL_PATH, RAW_DIR / 'hf_meta/questions.jsonl', normalized_path,
    RAW_DIR / 'hf_meta/code_stock.csv', structured_path, SUBMISSION_PACKAGE_DIR,
    top_k=10, max_retries=2,
    dense_index_dir=EVAL_DIR / 'reports' / 'dense_bge_m3_enriched_full_corpus',
    dense_hf_repo_id='BAAI/bge-m3',
)
print(json.dumps(full_report, ensure_ascii=False, indent=2))
assert full_report['n_completed'] == full_report['n_questions'], (
    f"Inference incomplete: inspect {SUBMISSION_PACKAGE_DIR / 'work/failures'} and rerun; "
    'successful predictions are reused.'
)

**Hybrid (BM25 + dense BGE-M3) retrieval is now the default for this submission run** (cell above) -- backfill_fusion confirmed safe on a real Kaggle run (CHANGE_LOG.md 2026-08-31 hybrid-fusion-backfill entry): it floors at bm25_top10 and can only add coverage, never regress it. No separate opt-in step needed any more.


In [ ]:
from submission.build_submission import build as build_submission
from submission.validate_submission import validate as validate_submission
official_questions = RAW_DIR / 'hf_meta/questions.jsonl'
submission_path = SUBMISSION_PACKAGE_DIR / 'submission.json'
records = build_submission(official_questions, SUBMISSION_PACKAGE_DIR / 'work/predictions', submission_path)
validation_report = validate_submission(submission_path, official_questions, SUBMISSION_PACKAGE_DIR)
(SUBMISSION_PACKAGE_DIR / 'validation_report.json').write_text(
    json.dumps(validation_report, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(validation_report, ensure_ascii=False, indent=2))
assert validation_report['valid'], validation_report['errors'][:20]
print(f'READY: {submission_path} ({len(records)} official questions)')